In [1]:
import zipfile
from pathlib import Path
import os

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
zip_path = Path("../raw/mtc_gtfs.zip")

print("File exists:", zip_path.exists())

if zip_path.exists():
    print("File size:", round(zip_path.stat().st_size / (1024 * 1024), 2), "MB")
else:
    print("ERROR: ZIP file not found. Check the folder and filename.")

File exists: True
File size: 29.37 MB


In [3]:
with zipfile.ZipFile(zip_path, "r") as archive:

    file_list = archive.namelist()

    print("TOTAL FILES:", len(file_list))
    print("\nFILES AVAILABLE IN DATASET:\n")

    for filename in file_list:
        print(filename)

TOTAL FILES: 6

FILES AVAILABLE IN DATASET:

static_mtc_gtfs_data/
static_mtc_gtfs_data/all_mtc_routes.geojson
static_mtc_gtfs_data/bus_terminus.geojson
static_mtc_gtfs_data/chennai.bus.gtfs.zip
static_mtc_gtfs_data/mtc_stops.geojson
static_mtc_gtfs_data/routes_polyline.geojson


In [4]:
from io import BytesIO
import zipfile

with zipfile.ZipFile(zip_path, "r") as outer_zip:

    inner_zip_path = next(
        name for name in outer_zip.namelist()
        if name.endswith("chennai.bus.gtfs.zip")
    )

    inner_zip_bytes = outer_zip.read(inner_zip_path)

    with zipfile.ZipFile(BytesIO(inner_zip_bytes), "r") as gtfs_zip:

        gtfs_files = gtfs_zip.namelist()

        print("INNER GTFS FILE COUNT:", len(gtfs_files))
        print("\nGTFS FILES:\n")

        for filename in gtfs_files:
            print(filename)

INNER GTFS FILE COUNT: 8

GTFS FILES:

chennai.bus.gtfs/
chennai.bus.gtfs/agency.txt
chennai.bus.gtfs/stop_times.txt
chennai.bus.gtfs/trips.txt
chennai.bus.gtfs/feed_info.txt
chennai.bus.gtfs/stops.txt
chennai.bus.gtfs/calendar.txt
chennai.bus.gtfs/routes.txt


In [5]:
import json

geojson_files = [
    "static_mtc_gtfs_data/all_mtc_routes.geojson",
    "static_mtc_gtfs_data/bus_terminus.geojson",
    "static_mtc_gtfs_data/mtc_stops.geojson",
    "static_mtc_gtfs_data/routes_polyline.geojson"
]

with zipfile.ZipFile(zip_path, "r") as outer_zip:

    for filename in geojson_files:

        print("\n" + "=" * 60)
        print("FILE:", filename)

        with outer_zip.open(filename) as file:
            data = json.load(file)

        print("TYPE:", data.get("type"))

        features = data.get("features", [])

        print("FEATURE COUNT:", len(features))

        geometry_types = {}

        for feature in features:
            geometry = feature.get("geometry") or {}
            geometry_type = geometry.get("type", "Missing")

            geometry_types[geometry_type] = (
                geometry_types.get(geometry_type, 0) + 1
            )

        print("GEOMETRY TYPES:", geometry_types)

        if features:
            print("PROPERTY COLUMNS:")
            print(list(features[0].get("properties", {}).keys()))

            print("\nFIRST FEATURE PROPERTIES:")
            print(features[0].get("properties"))


FILE: static_mtc_gtfs_data/all_mtc_routes.geojson
TYPE: FeatureCollection
FEATURE COUNT: 637
GEOMETRY TYPES: {'MultiLineString': 637}
PROPERTY COLUMNS:
['route_name', 'source', 'destinatio']

FIRST FEATURE PROPERTIES:
{'route_name': '1', 'source': 'Thiruvottiyur Bus Stand', 'destinatio': 'Thiruvanmiyur Depot'}

FILE: static_mtc_gtfs_data/bus_terminus.geojson
TYPE: FeatureCollection
FEATURE COUNT: 75
GEOMETRY TYPES: {'Point': 75}
PROPERTY COLUMNS:
['Name of th', 'Ownership']

FIRST FEATURE PROPERTIES:
{'Name of th': 'Tambaram Bus Terminus', 'Ownership': 'LB'}

FILE: static_mtc_gtfs_data/mtc_stops.geojson
TYPE: FeatureCollection
FEATURE COUNT: 5519
GEOMETRY TYPES: {'Point': 5519}
PROPERTY COLUMNS:
['ogc_fid', 'feature_type', 'route_id', 'route_short_name', 'route_long_name', 'route_type', 'reverse_route_id', 'stop_id', 'stop_name', 'location_type', 'routes_served', 'routes_served_names', 'num_routes', 'trip_count', 'source']

FIRST FEATURE PROPERTIES:
{'ogc_fid': 4776, 'feature_type': '

In [6]:
def collect_coordinates(obj, coordinates):
    if isinstance(obj, (list, tuple)):

        if (
            len(obj) >= 2
            and isinstance(obj[0], (int, float))
            and isinstance(obj[1], (int, float))
        ):
            coordinates.append((obj[0], obj[1]))

        else:
            for item in obj:
                collect_coordinates(item, coordinates)


with zipfile.ZipFile(zip_path, "r") as outer_zip:

    for filename in geojson_files:

        with outer_zip.open(filename) as file:
            data = json.load(file)

        coordinates = []
        collect_coordinates(data.get("features", []), coordinates)

        print("\nFILE:", filename)

        if coordinates:
            longitudes = [point[0] for point in coordinates]
            latitudes = [point[1] for point in coordinates]

            print("Coordinate count:", len(coordinates))
            print("Longitude range:", min(longitudes), "to", max(longitudes))
            print("Latitude range:", min(latitudes), "to", max(latitudes))

        else:
            print("No coordinates found.")


FILE: static_mtc_gtfs_data/all_mtc_routes.geojson
No coordinates found.

FILE: static_mtc_gtfs_data/bus_terminus.geojson
No coordinates found.

FILE: static_mtc_gtfs_data/mtc_stops.geojson
No coordinates found.

FILE: static_mtc_gtfs_data/routes_polyline.geojson
No coordinates found.


In [7]:
def collect_coordinates(obj, coordinates):

    if isinstance(obj, dict):

        for value in obj.values():
            collect_coordinates(value, coordinates)

    elif isinstance(obj, (list, tuple)):

        if (
            len(obj) >= 2
            and isinstance(obj[0], (int, float))
            and isinstance(obj[1], (int, float))
        ):
            coordinates.append((obj[0], obj[1]))

        else:
            for item in obj:
                collect_coordinates(item, coordinates)


with zipfile.ZipFile(zip_path, "r") as outer_zip:

    for filename in geojson_files:

        with outer_zip.open(filename) as file:
            data = json.load(file)

        coordinates = []

        for feature in data.get("features", []):

            geometry = feature.get("geometry")

            if geometry:
                collect_coordinates(
                    geometry.get("coordinates"),
                    coordinates
                )

        print("\nFILE:", filename)

        if coordinates:

            longitudes = [p[0] for p in coordinates]
            latitudes = [p[1] for p in coordinates]

            print("Coordinate count:", len(coordinates))
            print("First coordinate:", coordinates[0])
            print(
                "Longitude range:",
                min(longitudes),
                "to",
                max(longitudes)
            )
            print(
                "Latitude range:",
                min(latitudes),
                "to",
                max(latitudes)
            )

        else:
            print("No coordinates found.")


FILE: static_mtc_gtfs_data/all_mtc_routes.geojson
Coordinate count: 222364
First coordinate: (80.30608, 13.17511)
Longitude range: 79.80187 to 80.33586
Latitude range: 12.61472 to 13.49041

FILE: static_mtc_gtfs_data/bus_terminus.geojson
Coordinate count: 75
First coordinate: (80.11967476766709, 12.928706542953071)
Longitude range: 79.9083001830784 to 80.32081341520885
Latitude range: 12.617436574667416 to 13.308125000482733

FILE: static_mtc_gtfs_data/mtc_stops.geojson
Coordinate count: 5519
First coordinate: (79.980321079, 12.691013107)
Longitude range: 79.61503383 to 80.335835301
Latitude range: 12.614785534 to 13.489862372

FILE: static_mtc_gtfs_data/routes_polyline.geojson
Coordinate count: 709008
First coordinate: (80.25938, 12.9873)
Longitude range: 78.68087 to 80.33591
Latitude range: 10.78405 to 13.49041


In [8]:
import pandas as pd
from io import BytesIO

gtfs_data = {}

with zipfile.ZipFile(zip_path, "r") as outer_zip:

    inner_bytes = outer_zip.read(inner_zip_path)

    with zipfile.ZipFile(BytesIO(inner_bytes), "r") as gtfs_zip:

        for filename in gtfs_zip.namelist():

            if filename.endswith(".txt"):

                table_name = Path(filename).stem

                with gtfs_zip.open(filename) as file:

                    gtfs_data[table_name] = pd.read_csv(
                        file,
                        dtype=str
                    )

print("GTFS tables loaded:", list(gtfs_data.keys()))

for name, df in gtfs_data.items():

    print("\n" + "=" * 50)
    print("TABLE:", name)
    print("ROWS:", len(df))
    print("COLUMNS:", list(df.columns))
    print("\nFIRST 3 RECORDS:")
    display(df.head(3))

GTFS tables loaded: ['agency', 'stop_times', 'trips', 'feed_info', 'stops', 'calendar', 'routes']

TABLE: agency
ROWS: 1
COLUMNS: ['agency_id', 'agency_name', 'agency_url', 'agency_timezone', 'agency_lang', 'agency_phone', 'agency_fare_url']

FIRST 3 RECORDS:


,agency_id,agency_name,agency_url,agency_timezone,agency_lang,agency_phone,agency_fare_url
0,CUMTA,Chennai Unified Metropolitan Transport Authority,https://cumta.tn.gov.in/,Asia/Kolkata,en,NaN,NaN



TABLE: stop_times
ROWS: 1672859
COLUMNS: ['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence', 'stop_headsign']

FIRST 3 RECORDS:


,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign
0,4105-O-UP-28714,11:17:00,11:17:15,BpAiVnps,1,"{'fareStageNumber': '1', 'isStageStop': true}"
1,4105-O-UP-28714,11:19:15,11:19:30,fmQEzQWV,2,"{'fareStageNumber': '2', 'isStageStop': true}"
2,4105-O-UP-28714,11:21:30,11:21:45,IubGYoNi,3,2



TABLE: trips
ROWS: 68320
COLUMNS: ['route_id', 'service_id', 'trip_id', 'direction_id']

FIRST 3 RECORDS:


,route_id,service_id,trip_id,direction_id
0,4105,chennai_bus_service_1,4105-O-UP-28714,0
1,3055,chennai_bus_service_1,3055-O-UP-25711,0
2,3340,chennai_bus_service_1,3340-S-UP-137781,0


... (4 outputs left out)


TABLE: calendar
ROWS: 1
COLUMNS: ['service_id', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday', 'start_date', 'end_date']

FIRST 3 RECORDS:


,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,chennai_bus_service_1,1,1,1,1,1,1,1,20251028,20990101



TABLE: routes
ROWS: 4030
COLUMNS: ['route_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_type']

FIRST 3 RECORDS:


,route_id,agency_id,route_short_name,route_long_name,route_type
0,3,CUMTA,102,ADYAR B.T To SIRUSERI I.T PARK,3
1,7,CUMTA,102,BROADWAY To KELAMBAKKAM B.T,3
2,8,CUMTA,102,KELAMBAKKAM B.T To BROADWAY,3


In [9]:
with zipfile.ZipFile(zip_path, "r") as outer_zip:

    geo_stops = json.load(
        outer_zip.open(
            "static_mtc_gtfs_data/mtc_stops.geojson"
        )
    )

    geo_routes = json.load(
        outer_zip.open(
            "static_mtc_gtfs_data/routes_polyline.geojson"
        )
    )

gtfs_routes = set(
    gtfs_data["routes"]["route_id"].dropna()
)

gtfs_stops = set(
    gtfs_data["stops"]["stop_id"].dropna()
)

gtfs_trip_routes = set(
    gtfs_data["trips"]["route_id"].dropna()
)

geo_stop_ids = {
    str(f["properties"].get("stop_id"))
    for f in geo_stops["features"]
    if f.get("properties", {}).get("stop_id") is not None
}

geo_route_ids = {
    str(f["properties"].get("route_id"))
    for f in geo_routes["features"]
    if f.get("properties", {}).get("route_id") is not None
}

print("GTFS routes:", len(gtfs_routes))
print("GTFS stops:", len(gtfs_stops))
print("GTFS trip route IDs:", len(gtfs_trip_routes))

print("\nGeoJSON stop IDs:", len(geo_stop_ids))
print("GeoJSON route IDs:", len(geo_route_ids))

print("\nMatching stop IDs:",
      len(gtfs_stops.intersection(geo_stop_ids)))

print("Matching route IDs:",
      len(gtfs_routes.intersection(geo_route_ids)))

GTFS routes: 4030
GTFS stops: 7128
GTFS trip route IDs: 4030

GeoJSON stop IDs: 5519
GeoJSON route IDs: 1851

Matching stop IDs: 0
Matching route IDs: 0


In [11]:
# Inspect the flattened GeoJSON route data

geo_route_raw = pd.json_normalize(
    geo_routes["features"]
)

print("Total route features:", len(geo_route_raw))

print("\nAvailable columns:")
for col in geo_route_raw.columns:
    print(repr(col))

print("\nFirst route feature:")
display(geo_route_raw.head(1).T)

Total route features: 1851

Available columns:
'type'
'properties.route_id'
'properties.reverse_routeId'
'geometry.type'
'geometry.coordinates'

First route feature:


,0
type,Feature
properties.route_id,ntuHoTkS
properties.reverse_routeId,KFmySWzX
geometry.type,LineString
geometry.coordinates,"[[80.25938, 12.9873], [80.25817, 12.9874], [80..."


In [12]:
import json
from pathlib import Path

# Read all_mtc_routes.geojson from the original ZIP
with zipfile.ZipFile(zip_path, "r") as outer_zip:

    geo_route_name_data = json.loads(
        outer_zip.read(
            "static_mtc_gtfs_data/all_mtc_routes.geojson"
        )
    )

# Convert its features into a DataFrame
geo_route_name_df = pd.json_normalize(
    geo_route_name_data["features"]
)

print("Total route features:", len(geo_route_name_df))

print("\nAvailable columns:")
for col in geo_route_name_df.columns:
    print(col)

print("\nFirst three route records:")
display(geo_route_name_df.head(3))

Total route features: 637

Available columns:
type
properties.route_name
properties.source
properties.destinatio
geometry.type
geometry.coordinates

First three route records:


,type,properties.route_name,properties.source,properties.destinatio,geometry.type,geometry.coordinates
0,Feature,1,Thiruvottiyur Bus Stand,Thiruvanmiyur Depot,MultiLineString,"[[[80.30608, 13.17511], [80.30621, 13.17562], ..."
1,Feature,1B,Thiruvottiyur Bus Stand,Tambaram West Bus Stand,MultiLineString,"[[[80.30619, 13.1751], [80.30574, 13.17346], [..."
2,Feature,1C,Thiruvanmiyur Depot,Ennore Bus Depot,MultiLineString,"[[[80.25932, 12.98733], [80.25736, 12.98744], ..."


In [13]:
# Extract route information using the actual GeoJSON columns

geo_route_df = pd.DataFrame({
    "route_name": geo_route_name_df["properties.route_name"],
    "source": geo_route_name_df["properties.source"],
    "destination": geo_route_name_df["properties.destinatio"]
})

# Normalise route names
geo_route_df["route_normalized"] = (
    geo_route_df["route_name"].apply(normalize_name)
)

geo_route_df["long_normalized"] = (
    (
        geo_route_df["source"].fillna("").astype(str)
        + " TO "
        + geo_route_df["destination"].fillna("").astype(str)
    ).apply(normalize_name)
)

print("Unique geographic route names:",
      geo_route_df["route_normalized"].nunique())

print("\nSample geographic routes:")
display(geo_route_df.head(10))

Unique geographic route names: 637

Sample geographic routes:


,route_name,source,destination,route_normalized,long_normalized
0,1,Thiruvottiyur Bus Stand,Thiruvanmiyur Depot,1,THIRUVOTTIYUR BUS STAND TO THIRUVANMIYUR DEPOT
1,1B,Thiruvottiyur Bus Stand,Tambaram West Bus Stand,1B,THIRUVOTTIYUR BUS STAND TO TAMBARAM WEST BUS S...
2,1C,Thiruvanmiyur Depot,Ennore Bus Depot,1C,THIRUVANMIYUR DEPOT TO ENNORE BUS DEPOT
3,1D,Thiruvanmiyur Depot,Ennore Bus Depot,1D,THIRUVANMIYUR DEPOT TO ENNORE BUS DEPOT
4,2,Anna Square,Kaviarasu Kannadasan Nagar Bus Stand,2,ANNA SQUARE TO KAVIARASU KANNADASAN NAGAR BUS ...
5,2A,Kaviarasu Kannadasan Nagar Bus Stand,Anna Square,2A,KAVIARASU KANNADASAN NAGAR BUS STAND TO ANNA S...
6,2ACT,Anna Square,MGR Central Station,2ACT,ANNA SQUARE TO MGR CENTRAL STATION
7,3,Thiyagaraya Nagar Bus Depot,Thiruvanmiyur Depot,3,THIYAGARAYA NAGAR BUS DEPOT TO THIRUVANMIYUR D...
8,3CT,Tidel Park,Guindy Railway Station,3CT,TIDEL PARK TO GUINDY RAILWAY STATION
9,4,Broadway,Ennore Bus Depot,4,BROADWAY TO ENNORE BUS DEPOT


In [14]:
# Prepare GTFS route names
gtfs_route_df = gtfs_data["routes"].copy()

gtfs_route_df["short_normalized"] = (
    gtfs_route_df["route_short_name"]
    .apply(normalize_name)
)

gtfs_route_df["long_normalized"] = (
    gtfs_route_df["route_long_name"]
    .apply(normalize_name)
)

# Prepare geographic route names
geo_route_df["route_normalized"] = (
    geo_route_df["route_name"]
    .apply(normalize_name)
)

# Remove empty values before comparison
gtfs_short_names = set(
    gtfs_route_df["short_normalized"].dropna()
) - {""}

gtfs_long_names = set(
    gtfs_route_df["long_normalized"].dropna()
) - {""}

geo_short_names = set(
    geo_route_df["route_normalized"].dropna()
) - {""}

geo_long_names = set(
    geo_route_df["long_normalized"].dropna()
) - {""}

# Compare route numbers/names
short_matches = gtfs_short_names.intersection(
    geo_short_names
)

# Compare origin-destination names
long_matches = gtfs_long_names.intersection(
    geo_long_names
)

print("GTFS unique route short names:", len(gtfs_short_names))
print("GeoJSON unique route names:", len(geo_short_names))

print("\nExact route-name matches:", len(short_matches))
print("Exact origin-destination matches:", len(long_matches))

print("\nSample route-name matches:")
display(
    pd.DataFrame(
        sorted(short_matches),
        columns=["Matched route name"]
    ).head(30)
)

print("\nSample origin-destination matches:")
display(
    pd.DataFrame(
        sorted(long_matches),
        columns=["Matched origin-destination"]
    ).head(20)
)

GTFS unique route short names: 998
GeoJSON unique route names: 637

Exact route-name matches: 595
Exact origin-destination matches: 100

Sample route-name matches:


,Matched route name
0,1
1,101
2,102
3,102A
4,102C
5,102CT
6,102K
7,102P
8,102S
9,102X



Sample origin-destination matches:


,Matched origin-destination
0,AG DMS METRO STATION TO NUNGAMBAKKAM RAILWAY S...
1,AMARAMBEDU TO SAIDAPET
2,AMBATTUR INDUSTRIAL ESTATE TO THIRUMULLAIVOYAL...
3,AYYAPPAN MADAM TO BROADWAY
4,BROADWAY TO ALAMATHI
5,BROADWAY TO ERNAVOOR GATE
6,BROADWAY TO K K NAGAR
7,BROADWAY TO KADHAVOOR BUS STAND
8,BROADWAY TO KAMARAJ NAGAR
9,BROADWAY TO KORATTUR BUS TERMINUS


In [15]:
# Summarise GTFS routes by route number
gtfs_route_summary = (
    gtfs_route_df.groupby("short_normalized")
    .agg(
        gtfs_records=("route_id", "nunique"),
        gtfs_directions=("long_normalized", "nunique"),
        gtfs_examples=(
            "route_long_name",
            lambda x: " | ".join(
                x.dropna().astype(str).unique()[:3]
            )
        )
    )
    .reset_index()
)

# Summarise GeoJSON routes by route number
geo_route_summary = (
    geo_route_df.groupby("route_normalized")
    .agg(
        geo_features=("route_normalized", "size"),
        geo_directions=("long_normalized", "nunique"),
        geo_examples=(
            "long_normalized",
            lambda x: " | ".join(
                x.dropna().astype(str).unique()[:3]
            )
        )
    )
    .reset_index()
)

# Find common route numbers
route_comparison = gtfs_route_summary.merge(
    geo_route_summary,
    left_on="short_normalized",
    right_on="route_normalized",
    how="inner"
)

print("Common route numbers:", len(route_comparison))

print(
    "\nGTFS route numbers with multiple records:",
    (gtfs_route_summary["gtfs_records"] > 1).sum()
)

print(
    "GTFS route numbers with multiple directions:",
    (gtfs_route_summary["gtfs_directions"] > 1).sum()
)

print(
    "\nGeoJSON route numbers with multiple features:",
    (geo_route_summary["geo_features"] > 1).sum()
)

print(
    "GeoJSON route numbers with multiple directions:",
    (geo_route_summary["geo_directions"] > 1).sum()
)

print("\nSample route comparison:")
display(
    route_comparison[
        [
            "short_normalized",
            "gtfs_records",
            "gtfs_directions",
            "gtfs_examples",
            "geo_features",
            "geo_directions",
            "geo_examples"
        ]
    ].head(20)
)

print("\nRoutes needing closer inspection:")
display(
    route_comparison[
        (route_comparison["gtfs_directions"] > 1)
        | (route_comparison["geo_directions"] > 1)
    ].head(20)
)

Common route numbers: 595

GTFS route numbers with multiple records: 888
GTFS route numbers with multiple directions: 888

GeoJSON route numbers with multiple features: 0
GeoJSON route numbers with multiple directions: 0

Sample route comparison:


,short_normalized,gtfs_records,gtfs_directions,gtfs_examples,geo_features,geo_directions,geo_examples
0,1,8,8,THIRUVOTRIYUR B.T To THIRUVANMIYUR | M.G.R.CEN...,1,1,THIRUVOTTIYUR BUS STAND TO THIRUVANMIYUR DEPOT
1,101,11,11,BROADWAY To POONAMALLEE B.T | POONAMALLEE B.T ...,1,1,POONAMALLEE TERMINUS TO THIRUVOTTIYUR BUS STAND
2,102,18,18,ADYAR B.T To SIRUSERI I.T PARK | BROADWAY To K...,1,1,BROADWAY TO KELAMBAKKAM BUS TERMINAL
3,102A,2,2,PUDUPAKKAM OR VANDALOOR ROAD JUNCTION To THIRU...,1,1,PUDUPAKKAM OR VANDALOOR ROAD JUNCTION TO THIRU...
4,102C,7,7,BROADWAY To SEMMANCHERI BUS STAND | SEMMANCHER...,1,1,BROADWAY TO SEMMANCHERI BUS STAND
5,102CT,19,19,ADYAR B.T To SIRUSERI I.T PARK | ADYAR B.T To ...,1,1,SIRUCHERY I T PARK TO THIRUVANMIYUR DEPOT
6,102K,3,3,BROADWAY To KANNAGI NAGAR B.T | KANNAGI NAGAR ...,1,1,KANNAGI NAGAR BUS STAND TO BROADWAY
7,102P,20,20,GREAMS ROAD To BROADWAY | BROADWAY To Perumbak...,1,1,BROADWAY TO PERUMBAKKAM SCB
8,102S,5,5,KANNAGI NAGAR B.T To BROADWAY | BROADWAY To KE...,1,1,OKKIAM THORAIPAKM SECTOR COLONY TO BROADWAY
9,102X,14,14,BROADWAY To THIRUPORUR BUS STAND | THIRUPORUR ...,1,1,THIRUPORUR BUS STAND TO BROADWAY



Routes needing closer inspection:


,short_normalized,gtfs_records,gtfs_directions,gtfs_examples,route_normalized,geo_features,geo_directions,geo_examples
0,1,8,8,THIRUVOTRIYUR B.T To THIRUVANMIYUR | M.G.R.CEN...,1,1,1,THIRUVOTTIYUR BUS STAND TO THIRUVANMIYUR DEPOT
1,101,11,11,BROADWAY To POONAMALLEE B.T | POONAMALLEE B.T ...,101,1,1,POONAMALLEE TERMINUS TO THIRUVOTTIYUR BUS STAND
2,102,18,18,ADYAR B.T To SIRUSERI I.T PARK | BROADWAY To K...,102,1,1,BROADWAY TO KELAMBAKKAM BUS TERMINAL
3,102A,2,2,PUDUPAKKAM OR VANDALOOR ROAD JUNCTION To THIRU...,102A,1,1,PUDUPAKKAM OR VANDALOOR ROAD JUNCTION TO THIRU...
4,102C,7,7,BROADWAY To SEMMANCHERI BUS STAND | SEMMANCHER...,102C,1,1,BROADWAY TO SEMMANCHERI BUS STAND
5,102CT,19,19,ADYAR B.T To SIRUSERI I.T PARK | ADYAR B.T To ...,102CT,1,1,SIRUCHERY I T PARK TO THIRUVANMIYUR DEPOT
6,102K,3,3,BROADWAY To KANNAGI NAGAR B.T | KANNAGI NAGAR ...,102K,1,1,KANNAGI NAGAR BUS STAND TO BROADWAY
7,102P,20,20,GREAMS ROAD To BROADWAY | BROADWAY To Perumbak...,102P,1,1,BROADWAY TO PERUMBAKKAM SCB
8,102S,5,5,KANNAGI NAGAR B.T To BROADWAY | BROADWAY To KE...,102S,1,1,OKKIAM THORAIPAKM SECTOR COLONY TO BROADWAY
9,102X,14,14,BROADWAY To THIRUPORUR BUS STAND | THIRUPORUR ...,102X,1,1,THIRUPORUR BUS STAND TO BROADWAY


In [16]:
# Prepare GTFS stop records
gtfs_stop_df = gtfs_data["stops"].copy()

gtfs_stop_df["stop_normalized"] = (
    gtfs_stop_df["stop_name"].apply(normalize_name)
)

# Prepare geographic stop records
geo_stop_df = pd.json_normalize(
    geo_stops["features"]
)

geo_stop_df = pd.DataFrame({
    "stop_id": geo_stop_df["properties.stop_id"],
    "stop_name": geo_stop_df["properties.stop_name"],
    "routes_served": geo_stop_df["properties.routes_served"]
})

geo_stop_df["stop_normalized"] = (
    geo_stop_df["stop_name"].apply(normalize_name)
)

# Compare unique stop names
gtfs_stop_names = set(
    gtfs_stop_df["stop_normalized"].dropna()
) - {""}

geo_stop_names = set(
    geo_stop_df["stop_normalized"].dropna()
) - {""}

stop_matches = gtfs_stop_names.intersection(
    geo_stop_names
)

print("Unique GTFS stop names:", len(gtfs_stop_names))
print("Unique GeoJSON stop names:", len(geo_stop_names))
print("Exact stop-name matches:", len(stop_matches))

print("\nSample matching stop names:")
display(
    pd.DataFrame(
        sorted(stop_matches),
        columns=["Matched stop name"]
    ).head(30)
)

print("\nGTFS stop samples:")
display(gtfs_stop_df.head(10))

print("\nGeoJSON stop samples:")
display(geo_stop_df.head(10))

Unique GTFS stop names: 3648
Unique GeoJSON stop names: 2649
Exact stop-name matches: 1505

Sample matching stop names:


,Matched stop name
0,AALIYAMMAN TEMPLE OR TONDIARPET MASUTHI
1,ADAMBAKKAM
2,ADAMBAKKAM DEPOT
3,ADAMBAKKAM POLICE SATION
4,ADDUTHOTTI BRIDGE
5,ADHAM MARKET
6,ADHANUR
7,ADVENT CHRISTIAN CHURCH SEMBAKKAM
8,ADYAR DEPOT
9,ADYAR GANDHI NAGAR



GTFS stop samples:


,stop_id,stop_code,stop_name,stop_lat,stop_lon,location_type,stop_normalized
0,4,4,ADYAR B.T,12.549663,80.143925,0,ADYAR B T
1,3,3,ADYAR DEPOT,12.549663,80.143925,0,ADYAR DEPOT
2,174,174,S.R.P TOOLS,12.549663,80.143925,0,S R P TOOLS
3,83,83,KANDANCHAVADI,12.549663,80.143925,0,KANDANCHAVADI
4,194,194,THORAIPAKKAM TEA SHOP,12.549663,80.143925,0,THORAIPAKKAM TEA SHOP
5,116,116,MOOTAKARAN CHAVADI,12.549663,80.143925,0,MOOTAKARAN CHAVADI
6,85,85,KARAPAKKAM,12.549663,80.143925,0,KARAPAKKAM
7,168,168,SHOLINGANALLUR P.U.OFFICE,12.549663,80.143925,0,SHOLINGANALLUR P U OFFICE
8,100,100,KUMARAN NAGAR,12.549663,80.143925,0,KUMARAN NAGAR
9,165,165,SEMMANCHERI ALAMARAM,12.549663,80.143925,0,SEMMANCHERI ALAMARAM



GeoJSON stop samples:


,stop_id,stop_name,routes_served,stop_normalized
0,MTC_bInfxKVp,Chengalpattu New Bus Terminal,[],CHENGALPATTU NEW BUS TERMINAL
1,MTC_DdFWFvrk,Chengalpattu Bypass,[],CHENGALPATTU BYPASS
2,MTC_KcBICYpI,Chengalpattu Bypass,[],CHENGALPATTU BYPASS
3,MTC_qheIKvdP,Chengalpattu Old Bus Terminal,[],CHENGALPATTU OLD BUS TERMINAL
4,MTC_cCtcUCYS,Chengalpattu Old Bus Terminal,[],CHENGALPATTU OLD BUS TERMINAL
5,MTC_rBGGtpqI,Pulipakkam,[],PULIPAKKAM
6,MTC_RhHvrSFQ,Pulipakkam,[],PULIPAKKAM
7,MTC_GFyDwjlp,Rajakulipettai,[],RAJAKULIPETTAI
8,MTC_DeWEzUGk,Rajakulipettai,[],RAJAKULIPETTAI
9,MTC_YJlimzmz,Paranur Toll Plaza,[],PARANUR TOLL PLAZA


In [17]:
# Convert coordinates into numeric values
gtfs_stop_df["stop_lat"] = pd.to_numeric(
    gtfs_stop_df["stop_lat"],
    errors="coerce"
)

gtfs_stop_df["stop_lon"] = pd.to_numeric(
    gtfs_stop_df["stop_lon"],
    errors="coerce"
)

# Coordinate validity checks
valid_coordinates = (
    gtfs_stop_df["stop_lat"].between(-90, 90)
    & gtfs_stop_df["stop_lon"].between(-180, 180)
)

missing_coordinates = (
    gtfs_stop_df["stop_lat"].isna()
    | gtfs_stop_df["stop_lon"].isna()
)

print("Total GTFS stops:", len(gtfs_stop_df))
print("Valid coordinate records:", valid_coordinates.sum())
print("Missing coordinate records:", missing_coordinates.sum())
print(
    "Invalid coordinate records:",
    (~valid_coordinates & ~missing_coordinates).sum()
)

print("\nLatitude statistics:")
display(gtfs_stop_df["stop_lat"].describe())

print("\nLongitude statistics:")
display(gtfs_stop_df["stop_lon"].describe())

print("\nFirst 10 coordinate records:")
display(
    gtfs_stop_df[
        ["stop_id", "stop_name", "stop_lat", "stop_lon"]
    ].head(10)
)

Total GTFS stops: 7128
Valid coordinate records: 7128
Missing coordinate records: 0
Invalid coordinate records: 0

Latitude statistics:


count    7128.000000
mean       12.926965
std         0.236299
min        12.549663
25%        12.747545
50%        13.008560
75%        13.099743
max        13.444960
Name: stop_lat, dtype: float64


Longitude statistics:


count    7128.000000
mean       80.162952
std         0.081018
min        79.785160
25%        80.143030
50%        80.148090
75%        80.222020
max        80.335910
Name: stop_lon, dtype: float64


First 10 coordinate records:


,stop_id,stop_name,stop_lat,stop_lon
0,4,ADYAR B.T,12.549663,80.143925
1,3,ADYAR DEPOT,12.549663,80.143925
2,174,S.R.P TOOLS,12.549663,80.143925
3,83,KANDANCHAVADI,12.549663,80.143925
4,194,THORAIPAKKAM TEA SHOP,12.549663,80.143925
5,116,MOOTAKARAN CHAVADI,12.549663,80.143925
6,85,KARAPAKKAM,12.549663,80.143925
7,168,SHOLINGANALLUR P.U.OFFICE,12.549663,80.143925
8,100,KUMARAN NAGAR,12.549663,80.143925
9,165,SEMMANCHERI ALAMARAM,12.549663,80.143925


In [18]:
# Count unique coordinate pairs
unique_coordinates = gtfs_stop_df[
    ["stop_lat", "stop_lon"]
].drop_duplicates()

print("Total GTFS stops:", len(gtfs_stop_df))
print("Unique coordinate pairs:", len(unique_coordinates))
print(
    "Stops sharing coordinates:",
    len(gtfs_stop_df) - len(unique_coordinates)
)

# Find coordinates assigned to multiple stops
coordinate_groups = (
    gtfs_stop_df.groupby(["stop_lat", "stop_lon"])
    .agg(
        stop_count=("stop_id", "count"),
        stop_names=("stop_name", lambda x: " | ".join(
            x.astype(str).head(8)
        )),
        stop_ids=("stop_id", lambda x: " | ".join(
            x.astype(str).head(8)
        ))
    )
    .reset_index()
    .sort_values("stop_count", ascending=False)
)

print("\nTop 20 most frequently repeated coordinates:")
display(coordinate_groups.head(20))

print("\nCoordinate duplication distribution:")
print(
    coordinate_groups["stop_count"]
    .value_counts()
    .sort_index()
    .to_string()
)

Total GTFS stops: 7128
Unique coordinate pairs: 5123
Stops sharing coordinates: 2005

Top 20 most frequently repeated coordinates:


,stop_lat,stop_lon,stop_count,stop_names,stop_ids
0,12.549663,80.143925,1634,ADYAR B.T | ADYAR DEPOT | S.R.P TOOLS | KANDAN...,4 | 3 | 174 | 83 | 194 | 116 | 85 | 168
3713,13.115320,80.147850,4,SIKKARAYAPURAM UZHAVAR SANTHAI | SIKKARAYAPURA...,3772fb9352d630ff467f90a8a2e97dd3 | 35805cf45b3...
2215,13.034330,80.235580,3,THYAGARAYA NAGAR BURKIT ROAD | THYAGARAYA NAGA...,NFETpHWK | YcfPQopp | 75cdae9b
2076,13.026380,80.265410,3,MANDAVELI B.T | MANDAVELI | MANDAVELI B.T,YmDpkIHQ | ALSylDBQ | 48f74d4b
2831,13.065460,80.283640,3,MADRAS UNIVERSITY | MADRAS UNIVERSITY | MADRAS...,pSVDMeCH | tqWpBqnC | 52c0eeee
3445,13.100190,80.215770,3,ICF | ICF | ICF,TcTonWJL | mCHzTQKa | 64d33131
3632,13.110250,80.152250,3,AMBATTUR CANARA BANK | AMBATTUR CANARA BANK | ...,00ca43ed43b0b348ca41b792450f98b5 | 766f136a108...
516,12.862120,80.073570,2,URAPAKKAM ANANTHAVALLI SCHOOL | URAPAKKAM ANAN...,NxbXiMUL | 1edc3f19
1913,13.016550,80.189910,2,TRADE CENTRE NANDAMBAKKAM | TRADE CENTRE NANDA...,rBCwdRqs | a440fceb
2193,13.032750,80.277550,2,SANTHOME CHURCH | SANTHOME CHURCH,be6ee9cf70a4355ebf31eaa5fce764de | c374c130c16...



Coordinate duplication distribution:
stop_count
1       4757
2        359
3          5
4          1
1634       1


In [19]:
# Extract GeoJSON stop coordinates
geo_stop_features = geo_stops["features"]

geo_coordinate_records = []

for feature in geo_stop_features:
    properties = feature.get("properties", {})
    geometry = feature.get("geometry", {})

    coordinates = geometry.get("coordinates", [])

    if (
        geometry.get("type") == "Point"
        and len(coordinates) >= 2
    ):
        geo_coordinate_records.append({
            "stop_id": properties.get("stop_id"),
            "stop_name": properties.get("stop_name"),
            "longitude": coordinates[0],
            "latitude": coordinates[1]
        })

geo_coordinates_df = pd.DataFrame(
    geo_coordinate_records
)

# Convert coordinates to numeric
geo_coordinates_df["latitude"] = pd.to_numeric(
    geo_coordinates_df["latitude"],
    errors="coerce"
)

geo_coordinates_df["longitude"] = pd.to_numeric(
    geo_coordinates_df["longitude"],
    errors="coerce"
)

print("Total GeoJSON point records:", len(geo_coordinates_df))

print(
    "Valid coordinate records:",
    geo_coordinates_df[
        ["latitude", "longitude"]
    ].notna().all(axis=1).sum()
)

print(
    "Unique coordinate pairs:",
    geo_coordinates_df[
        ["latitude", "longitude"]
    ].drop_duplicates().shape[0]
)

print("\nGeoJSON coordinate statistics:")
display(
    geo_coordinates_df[
        ["latitude", "longitude"]
    ].describe()
)

print("\nFirst 15 GeoJSON coordinate records:")
display(geo_coordinates_df.head(15))

Total GeoJSON point records: 5519
Valid coordinate records: 5519
Unique coordinate pairs: 5518

GeoJSON coordinate statistics:


,latitude,longitude
count,5519.000000,5519.000000
mean,13.055067,80.161897
std,0.145606,0.096388
min,12.614786,79.615034
25%,12.970633,80.108266
50%,13.056485,80.182218
75%,13.131767,80.233966
max,13.489862,80.335835



First 15 GeoJSON coordinate records:


,stop_id,stop_name,longitude,latitude
0,MTC_bInfxKVp,Chengalpattu New Bus Terminal,79.980321,12.691013
1,MTC_DdFWFvrk,Chengalpattu Bypass,79.966839,12.697138
2,MTC_KcBICYpI,Chengalpattu Bypass,79.967000,12.697200
3,MTC_qheIKvdP,Chengalpattu Old Bus Terminal,79.976811,12.696941
4,MTC_cCtcUCYS,Chengalpattu Old Bus Terminal,79.976703,12.696930
5,MTC_rBGGtpqI,Pulipakkam,79.979030,12.711444
6,MTC_RhHvrSFQ,Pulipakkam,79.978770,12.711532
7,MTC_GFyDwjlp,Rajakulipettai,79.980586,12.716496
8,MTC_DeWEzUGk,Rajakulipettai,79.980898,12.716841
9,MTC_YJlimzmz,Paranur Toll Plaza,79.981779,12.724178


In [20]:
import numpy as np

# Prepare geographic stop coordinates
geo_coordinates_df["stop_normalized"] = (
    geo_coordinates_df["stop_name"].apply(normalize_name)
)

# Remove records without names or coordinates
gtfs_valid = gtfs_stop_df.dropna(
    subset=["stop_normalized", "stop_lat", "stop_lon"]
).copy()

geo_valid = geo_coordinates_df.dropna(
    subset=["stop_normalized", "latitude", "longitude"]
).copy()

# Match stops by normalized name
matched_stops = gtfs_valid.merge(
    geo_valid,
    on="stop_normalized",
    how="inner",
    suffixes=("_gtfs", "_geo")
)

# Haversine distance calculation
def haversine_distance(lat1, lon1, lat2, lon2):
    earth_radius = 6371000  # metres

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    return 2 * earth_radius * np.arcsin(
        np.sqrt(np.clip(a, 0, 1))
    )

# Calculate distance for each matching pair
matched_stops["distance_metres"] = haversine_distance(
    matched_stops["stop_lat"],
    matched_stops["stop_lon"],
    matched_stops["latitude"],
    matched_stops["longitude"]
)

print("GTFS stop records:", len(gtfs_valid))
print("GeoJSON stop records:", len(geo_valid))
print("Matching stop-name pairs:", len(matched_stops))

# Find the nearest GeoJSON match for each GTFS stop
nearest_matches = (
    matched_stops
    .sort_values("distance_metres")
    .drop_duplicates(subset="stop_id_gtfs", keep="first")
)

print("\nUnique GTFS stops with a matching name:",
      nearest_matches["stop_id_gtfs"].nunique())

print("\nNearest-match distance statistics:")
display(nearest_matches["distance_metres"].describe())

# Distance categories
distance_ranges = {
    "Within 50 metres": nearest_matches["distance_metres"] <= 50,
    "Within 100 metres": nearest_matches["distance_metres"] <= 100,
    "Within 500 metres": nearest_matches["distance_metres"] <= 500,
    "Within 1 kilometre": nearest_matches["distance_metres"] <= 1000,
    "More than 1 kilometre": nearest_matches["distance_metres"] > 1000
}

print("\nNearest-match distance classification:")

for label, condition in distance_ranges.items():
    print(f"{label}: {condition.sum()}")

print("\nSample nearest matches:")
display(
    nearest_matches[
        [
            "stop_id_gtfs",
            "stop_name_gtfs",
            "stop_id_geo",
            "stop_name_geo",
            "distance_metres"
        ]
    ].head(20)
)

print("\nGTFS stops using the suspicious repeated coordinate:")
display(
    gtfs_stop_df[
        (gtfs_stop_df["stop_lat"] == 12.549663)
        & (gtfs_stop_df["stop_lon"] == 80.143925)
    ][["stop_id", "stop_name"]].head(20)
)

GTFS stop records: 7128
GeoJSON stop records: 5519
Matching stop-name pairs: 8461

Unique GTFS stops with a matching name: 3557

Nearest-match distance statistics:


count      3557.000000
mean       7658.531363
std       20356.862411
min           0.000000
25%           2.625283
50%           5.298183
75%          12.405303
max      104637.515841
Name: distance_metres, dtype: float64


Nearest-match distance classification:
Within 50 metres: 2920
Within 100 metres: 2951
Within 500 metres: 3048
Within 1 kilometre: 3064
More than 1 kilometre: 493

Sample nearest matches:


,stop_id_gtfs,stop_name_gtfs,stop_id_geo,stop_name_geo,distance_metres
7928,uqWvvTiu,TONDIYARPET HIGH ROAD,MTC_uqWvvTiu,Tondiyarpet High Road,0.000000
8222,aNqhHdpI,MANIKAM NAGAR JUNCTION,MTC_aNqhHdpI,Manikam Nagar Junction,0.000000
69,wqGWmwVd,KARAPAKKAM,MTC_wqGWmwVd,Karapakkam,0.025335
3889,oaiHprKy,PADIANALLUR CHECK POST OR VIJAYA NALLUR,MTC_oaiHprKy,Padianallur Check Post Or Vijaya Nallur,0.039794
7431,WneZtuRq,VALLAKOTTAI,MTC_WneZtuRq,Vallakottai,0.056228
2308,yXKTuRtI,METTUPALAYAM,MTC_yXKTuRtI,Mettupalayam,0.065573
3511,CYSggHNR,RAILWAY QUARTERS,MTC_CYSggHNR,Railway Quarters,0.071846
7318,eJIFTpTd,ORAGADAM ROAD JUNCTION,MTC_eJIFTpTd,Oragadam Road Junction,0.081245
311,zKpOsJgw,ANANDBAGH,MTC_zKpOsJgw,Anandbagh,0.087571
7897,aFLknQpO,COWL BAZAR,MTC_aFLknQpO,Cowl Bazar,0.110200



GTFS stops using the suspicious repeated coordinate:


,stop_id,stop_name
0,4,ADYAR B.T
1,3,ADYAR DEPOT
2,174,S.R.P TOOLS
3,83,KANDANCHAVADI
4,194,THORAIPAKKAM TEA SHOP
5,116,MOOTAKARAN CHAVADI
6,85,KARAPAKKAM
7,168,SHOLINGANALLUR P.U.OFFICE
8,100,KUMARAN NAGAR
9,165,SEMMANCHERI ALAMARAM


In [22]:
import numpy as np
import pandas as pd

# Prepare matching identifiers
gtfs_id_check = gtfs_stop_df.copy()
geo_id_check = geo_coordinates_df.copy()

gtfs_id_check["stop_id_match"] = (
    gtfs_id_check["stop_id"].astype(str).str.strip()
)

geo_id_check["stop_id_match"] = (
    geo_id_check["stop_id"]
    .astype(str)
    .str.strip()
    .str.replace(r"^MTC_", "", regex=True)
)

# Check duplicate identifiers before merging
print("GTFS duplicate stop IDs:",
      gtfs_id_check["stop_id_match"].duplicated().sum())

print("GeoJSON duplicate stop IDs:",
      geo_id_check["stop_id_match"].duplicated().sum())

# Keep one record per ID for this initial matching test
gtfs_id_unique = gtfs_id_check.drop_duplicates(
    subset="stop_id_match"
)

geo_id_unique = geo_id_check.drop_duplicates(
    subset="stop_id_match"
)

# Match by identifier
id_matches = gtfs_id_unique.merge(
    geo_id_unique,
    on="stop_id_match",
    how="inner",
    suffixes=("_gtfs", "_geo")
)

print("\nGTFS unique stop IDs:", len(gtfs_id_unique))
print("GeoJSON unique stop IDs:", len(geo_id_unique))
print("Exact ID matches:", len(id_matches))

# Calculate geographic distance
id_matches["distance_metres"] = haversine_distance(
    id_matches["stop_lat"].astype(float),
    id_matches["stop_lon"].astype(float),
    id_matches["latitude"].astype(float),
    id_matches["longitude"].astype(float)
)

print("\nID-match distance statistics:")
print(id_matches["distance_metres"].describe())

print("\nID-match distance classification:")
print("Within 50 metres:",
      (id_matches["distance_metres"] <= 50).sum())
print("Within 100 metres:",
      (id_matches["distance_metres"] <= 100).sum())
print("Within 500 metres:",
      (id_matches["distance_metres"] <= 500).sum())
print("Within 1 kilometre:",
      (id_matches["distance_metres"] <= 1000).sum())
print("More than 1 kilometre:",
      (id_matches["distance_metres"] > 1000).sum())

print("\nSample ID matches:")
display(
    id_matches[
        [
            "stop_id_match",
            "stop_name_gtfs",
            "stop_name_geo",
            "distance_metres"
        ]
    ].head(20)
)

GTFS duplicate stop IDs: 0
GeoJSON duplicate stop IDs: 0

GTFS unique stop IDs: 7128
GeoJSON unique stop IDs: 5519
Exact ID matches: 4209

ID-match distance statistics:
count    4209.000000
mean        4.987726
std         4.467118
min         0.000000
25%         2.047330
50%         3.923642
75%         6.800026
max        69.789321
Name: distance_metres, dtype: float64

ID-match distance classification:
Within 50 metres: 4205
Within 100 metres: 4209
Within 500 metres: 4209
Within 1 kilometre: 4209
More than 1 kilometre: 0

Sample ID matches:


,stop_id_match,stop_name_gtfs,stop_name_geo,distance_metres
0,cGoHBmgo,HIGH COURT,High Court,6.335008
1,fwfbiPwk,PARRYS CORNER,Parrys Corner,2.080166
2,mgTKojDA,R B I PARRYS,R B I Parrys,8.411081
3,drmArhka,SECRETARIAT,Secretariat,5.816329
4,XypZkLUe,WAR MEMORIAL,War Memorial,6.600778
5,gaLZwYvS,ANNA SQUARE B.T,Anna Square,10.334471
6,qFzqwYZZ,CHEPPAKAM,Marina Beach,4.126279
7,MBTcSIip,KANNAGI STATUE OR PRESIDENCY COLLEGE,Kannagi Statue Or Presidency College,4.882739
8,ppabfQGz,VIVEKANANDA HOUSE BUS TERMINUS,Swami Vivekananda House,6.908146
9,rloRqLNl,TRIPLICANE QUEEN MARYS COLLEGE,Queen Marys College,10.157954


In [23]:
# Identify GTFS stops without an exact GeoJSON ID match
matched_ids = set(id_matches["stop_id_match"])

unmatched_gtfs = gtfs_id_unique[
    ~gtfs_id_unique["stop_id_match"].isin(matched_ids)
].copy()

matched_gtfs = gtfs_id_unique[
    gtfs_id_unique["stop_id_match"].isin(matched_ids)
].copy()

print("Matched GTFS stops:", len(matched_gtfs))
print("Unmatched GTFS stops:", len(unmatched_gtfs))

# Check whether unmatched stops have repeated coordinates
unmatched_coordinate_counts = (
    unmatched_gtfs.groupby(["stop_lat", "stop_lon"])
    .size()
    .reset_index(name="stop_count")
    .sort_values("stop_count", ascending=False)
)

print("\nUnmatched stops sharing coordinates:")
print(
    "Stops belonging to repeated-coordinate groups:",
    unmatched_coordinate_counts.loc[
        unmatched_coordinate_counts["stop_count"] > 1,
        "stop_count"
    ].sum()
)

print("\nLargest repeated-coordinate groups among unmatched stops:")
display(unmatched_coordinate_counts.head(15))

# Compare coordinate uniqueness between matched and unmatched groups
print("\nUnique coordinate pairs:")
print(
    "Matched GTFS:",
    matched_gtfs[["stop_lat", "stop_lon"]].drop_duplicates().shape[0]
)
print(
    "Unmatched GTFS:",
    unmatched_gtfs[["stop_lat", "stop_lon"]].drop_duplicates().shape[0]
)

# Inspect the four ID matches with distances above 50 metres
print("\nID matches with distance above 50 metres:")
display(
    id_matches[id_matches["distance_metres"] > 50][
        [
            "stop_id_match",
            "stop_name_gtfs",
            "stop_name_geo",
            "distance_metres"
        ]
    ].sort_values("distance_metres", ascending=False)
)

# Inspect sample unmatched GTFS records
print("\nSample unmatched GTFS stops:")
display(
    unmatched_gtfs[
        ["stop_id", "stop_name", "stop_lat", "stop_lon"]
    ].head(25)
)

Matched GTFS stops: 4209
Unmatched GTFS stops: 2919

Unmatched stops sharing coordinates:
Stops belonging to repeated-coordinate groups: 1709

Largest repeated-coordinate groups among unmatched stops:


,stop_lat,stop_lon,stop_count
0,12.549663,80.143925,1634
961,13.115320,80.147850,4
936,13.110250,80.152250,3
1224,13.345410,80.238620,2
524,13.037700,80.264290,2
356,12.997430,80.096320,2
128,12.862080,80.073550,2
129,12.862090,80.073350,2
495,13.032750,80.267220,2
171,12.909000,80.098340,2



Unique coordinate pairs:
Matched GTFS: 4162
Unmatched GTFS: 1247

ID matches with distance above 50 metres:


,stop_id_match,stop_name_gtfs,stop_name_geo,distance_metres
2665,ufLmazVu,PERAMBUR RAILWAY STATION,Perambur Railway Station,69.789321
2178,WKWTpoqQ,M.M.B.T,MMBT,60.771815
269,BpAiVnps,M.G.R.KOYAMBEDU B.T,CMBT Koyembedu Bus Depot,60.486964
751,gyQqEnFq,THIRUVANMIYUR B.T,Thiruvanmiyur Depot,50.032848



Sample unmatched GTFS stops:


,stop_id,stop_name,stop_lat,stop_lon
0,4,ADYAR B.T,12.549663,80.143925
1,3,ADYAR DEPOT,12.549663,80.143925
2,174,S.R.P TOOLS,12.549663,80.143925
3,83,KANDANCHAVADI,12.549663,80.143925
4,194,THORAIPAKKAM TEA SHOP,12.549663,80.143925
5,116,MOOTAKARAN CHAVADI,12.549663,80.143925
6,85,KARAPAKKAM,12.549663,80.143925
7,168,SHOLINGANALLUR P.U.OFFICE,12.549663,80.143925
8,100,KUMARAN NAGAR,12.549663,80.143925
9,165,SEMMANCHERI ALAMARAM,12.549663,80.143925


In [24]:
# Work on a copy; do not modify the original GTFS table
unmatched_quality = unmatched_gtfs.copy()

# Count how many unmatched stops share each coordinate
unmatched_quality["coordinate_group_size"] = (
    unmatched_quality
    .groupby(["stop_lat", "stop_lon"])["stop_id"]
    .transform("count")
)

# Classify coordinate quality
dominant_lat = 12.549663
dominant_lon = 80.143925

is_dominant = (
    np.isclose(unmatched_quality["stop_lat"].astype(float), dominant_lat)
    & np.isclose(unmatched_quality["stop_lon"].astype(float), dominant_lon)
)

unmatched_quality["coordinate_status"] = np.select(
    [
        is_dominant,
        unmatched_quality["coordinate_group_size"] > 1
    ],
    [
        "placeholder_candidate",
        "repeated_coordinate_review"
    ],
    default="unique_unmatched_provisional"
)

print("Coordinate quality classification:")
print(
    unmatched_quality["coordinate_status"]
    .value_counts()
)

print("\nCoordinate group-size summary:")
print(
    unmatched_quality["coordinate_group_size"]
    .describe()
)

print("\nSample unique unmatched stops:")
display(
    unmatched_quality[
        unmatched_quality["coordinate_status"]
        == "unique_unmatched_provisional"
    ][
        ["stop_id", "stop_name", "stop_lat", "stop_lon"]
    ].head(20)
)

print("\nOther repeated-coordinate groups:")
display(
    unmatched_quality[
        (unmatched_quality["coordinate_group_size"] > 1)
        & (~is_dominant)
    ][
        ["stop_lat", "stop_lon", "coordinate_group_size"]
    ].drop_duplicates()
    .sort_values("coordinate_group_size", ascending=False)
)

Coordinate quality classification:
coordinate_status
placeholder_candidate           1634
unique_unmatched_provisional    1210
repeated_coordinate_review        75
Name: count, dtype: int64

Coordinate group-size summary:
count    2919.000000
mean      915.151422
std       810.748767
min         1.000000
25%         1.000000
50%      1634.000000
75%      1634.000000
max      1634.000000
Name: coordinate_group_size, dtype: float64

Sample unique unmatched stops:


,stop_id,stop_name,stop_lat,stop_lon
18,fa11bdb91361aafff55ff4db66c424d2,BROADWAY,13.08703,80.28440
29,226fa83f86509c4095f3e613d4ccaf6d,ALL INDIA RADIO LIGHT HOUSE,13.03904,80.28025
33,ce3115c68314873cf18631cbedbc1fe3,SATHYA STUDIO,13.01611,80.25968
35,9ced87eb513fd55497cdf58b974e8309,ADYAR TELEPHONE EXCHANGE,13.00136,80.25664
38,ac8933babf11be7381b34ac152ee95ba,L.B ROAD,12.98625,80.25556
63,b8005a7a060e5deecbd69f664c0bdd6a,KAZHIPATTUR,12.81965,80.23053
64,741e781007f44f721507d7fbf909e032,PADUR AKSHAYA,12.81098,80.22808
72,393dc1f9ae920ec7d127cd7c8d3c31ab,CHETTINAD HOSPITAL,12.79611,80.22290
88,ddf9b7895d25353b687b012d51771836,P U OFFICE SHOLINGANALLUR,12.90205,80.22794
104,f166c5c0ee6d7f8d330953579f23b488,ADYAR SIGNAL,13.00723,80.25838



Other repeated-coordinate groups:


,stop_lat,stop_lon,coordinate_group_size
2108,13.11532,80.14785,4
2107,13.11025,80.15225,3
158,12.98855,80.25589,2
160,13.01603,80.25964,2
411,13.06275,80.19713,2
163,13.03275,80.27755,2
531,13.03275,80.26722,2
532,13.03770,80.26429,2
750,13.10014,80.25171,2
845,13.08532,80.20120,2


In [25]:
# Step 17A: Create a prepared stop dataset

# Start from all unique GTFS stop records
prepared_stops = gtfs_id_unique.copy()

# Attach GeoJSON information for exact ID matches
geo_match_columns = [
    "stop_id_match",
    "stop_id_geo",
    "stop_name_geo",
    "latitude",
    "longitude",
    "distance_metres"
]

prepared_stops = prepared_stops.merge(
    id_matches[geo_match_columns],
    on="stop_id_match",
    how="left"
)

# Identify matched stops
is_matched = prepared_stops["stop_id_geo"].notna()

# Identify the dominant suspicious coordinate
is_placeholder = (
    np.isclose(prepared_stops["stop_lat"].astype(float), 12.549663)
    & np.isclose(prepared_stops["stop_lon"].astype(float), 80.143925)
)

# Calculate coordinate group sizes across all unmatched stops
unmatched_coordinate_counts = (
    unmatched_gtfs.groupby(["stop_lat", "stop_lon"])
    .size()
)

coordinate_keys = list(zip(
    prepared_stops["stop_lat"],
    prepared_stops["stop_lon"]
))

prepared_stops["coordinate_group_size"] = [
    unmatched_coordinate_counts.get(key, 0)
    if not matched else 1
    for key, matched in zip(coordinate_keys, is_matched)
]

is_repeated_review = (
    (~is_matched)
    & (~is_placeholder)
    & (prepared_stops["coordinate_group_size"] > 1)
)

# Assign quality classification
prepared_stops["coordinate_status"] = np.select(
    [
        is_matched,
        is_placeholder,
        is_repeated_review
    ],
    [
        "verified_geojson_match",
        "placeholder_candidate",
        "repeated_coordinate_review"
    ],
    default="unique_unmatched_provisional"
)

# Preserve original GTFS coordinates and select map coordinates
prepared_stops["map_lat"] = np.nan
prepared_stops["map_lon"] = np.nan

# Matched records use GeoJSON coordinates
prepared_stops.loc[is_matched, "map_lat"] = (
    prepared_stops.loc[is_matched, "latitude"]
)

prepared_stops.loc[is_matched, "map_lon"] = (
    prepared_stops.loc[is_matched, "longitude"]
)

# Unique unmatched records use provisional GTFS coordinates
is_provisional = (
    prepared_stops["coordinate_status"]
    == "unique_unmatched_provisional"
)

prepared_stops.loc[is_provisional, "map_lat"] = (
    prepared_stops.loc[is_provisional, "stop_lat"]
)

prepared_stops.loc[is_provisional, "map_lon"] = (
    prepared_stops.loc[is_provisional, "stop_lon"]
)

# Mark whether the stop is included in the default map
prepared_stops["include_in_default_map"] = (
    prepared_stops["map_lat"].notna()
    & prepared_stops["map_lon"].notna()
)

print("Prepared stop dataset:", prepared_stops.shape)

print("\nCoordinate status counts:")
print(prepared_stops["coordinate_status"].value_counts())

print("\nDefault map inclusion:")
print(prepared_stops["include_in_default_map"].value_counts())

print("\nMissing selected map coordinates:")
print(
    prepared_stops[["map_lat", "map_lon"]]
    .isna()
    .any(axis=1)
    .sum()
)

print("\nPrepared dataset sample:")
display(
    prepared_stops[
        [
            "stop_id",
            "stop_name",
            "coordinate_status",
            "stop_lat",
            "stop_lon",
            "map_lat",
            "map_lon",
            "distance_metres",
            "include_in_default_map"
        ]
    ].head(20)
)

Prepared stop dataset: (7128, 18)

Coordinate status counts:
coordinate_status
verified_geojson_match          4209
placeholder_candidate           1634
unique_unmatched_provisional    1210
repeated_coordinate_review        75
Name: count, dtype: int64

Default map inclusion:
include_in_default_map
True     5419
False    1709
Name: count, dtype: int64

Missing selected map coordinates:
1709

Prepared dataset sample:


,stop_id,stop_name,coordinate_status,stop_lat,stop_lon,map_lat,map_lon,distance_metres,include_in_default_map
0,4,ADYAR B.T,placeholder_candidate,12.549663,80.143925,NaN,NaN,NaN,False
1,3,ADYAR DEPOT,placeholder_candidate,12.549663,80.143925,NaN,NaN,NaN,False
2,174,S.R.P TOOLS,placeholder_candidate,12.549663,80.143925,NaN,NaN,NaN,False
3,83,KANDANCHAVADI,placeholder_candidate,12.549663,80.143925,NaN,NaN,NaN,False
4,194,THORAIPAKKAM TEA SHOP,placeholder_candidate,12.549663,80.143925,NaN,NaN,NaN,False
5,116,MOOTAKARAN CHAVADI,placeholder_candidate,12.549663,80.143925,NaN,NaN,NaN,False
6,85,KARAPAKKAM,placeholder_candidate,12.549663,80.143925,NaN,NaN,NaN,False
7,168,SHOLINGANALLUR P.U.OFFICE,placeholder_candidate,12.549663,80.143925,NaN,NaN,NaN,False
8,100,KUMARAN NAGAR,placeholder_candidate,12.549663,80.143925,NaN,NaN,NaN,False
9,165,SEMMANCHERI ALAMARAM,placeholder_candidate,12.549663,80.143925,NaN,NaN,NaN,False


In [26]:
from pathlib import Path
import json

# Create output directory
output_dir = Path("../prepared")
output_dir.mkdir(parents=True, exist_ok=True)

# Select and order the columns for the complete prepared dataset
prepared_columns = [
    "stop_id",
    "stop_name",
    "stop_id_match",
    "stop_id_geo",
    "stop_name_geo",
    "stop_lat",
    "stop_lon",
    "latitude",
    "longitude",
    "map_lat",
    "map_lon",
    "distance_metres",
    "coordinate_group_size",
    "coordinate_status",
    "include_in_default_map"
]

export_stops = prepared_stops[prepared_columns].copy()

# Keep identifiers as strings
export_stops["stop_id"] = export_stops["stop_id"].astype(str)
export_stops["stop_id_match"] = (
    export_stops["stop_id_match"].astype(str)
)

# Create the default map dataset
stops_for_map = export_stops[
    export_stops["include_in_default_map"]
].copy()

# Create a quality summary
quality_summary = (
    export_stops.groupby("coordinate_status")
    .agg(
        stop_count=("stop_id", "count"),
        map_included=("include_in_default_map", "sum")
    )
    .reset_index()
)

# Validation checks before export
assert len(export_stops) == 7128
assert export_stops["stop_id"].is_unique
assert len(stops_for_map) == 5419
assert export_stops["map_lat"].notna().sum() == 5419
assert export_stops["map_lon"].notna().sum() == 5419

assert (
    export_stops.loc[
        export_stops["coordinate_status"] == "placeholder_candidate",
        "map_lat"
    ].isna().all()
)

# Export CSV files
export_stops.to_csv(
    output_dir / "prepared_stops.csv",
    index=False,
    encoding="utf-8"
)

stops_for_map.to_csv(
    output_dir / "stops_for_map.csv",
    index=False,
    encoding="utf-8"
)

quality_summary.to_csv(
    output_dir / "stop_quality_summary.csv",
    index=False,
    encoding="utf-8"
)

# Export preparation metadata
metadata = {
    "dataset": "Chennai MTC prepared stop reference data",
    "source_files": [
        "chennai.bus.gtfs.zip",
        "mtc_stops.geojson"
    ],
    "coordinate_reference_system": "WGS84 (EPSG:4326)",
    "total_gtfs_stops": 7128,
    "default_map_stops": 5419,
    "coordinate_quality_rules": {
        "verified_geojson_match": (
            "Exact stop ID match after removing the MTC_ prefix; "
            "GeoJSON coordinates selected."
        ),
        "placeholder_candidate": (
            "GTFS coordinate 12.549663, 80.143925; "
            "excluded from default map."
        ),
        "repeated_coordinate_review": (
            "Unmatched stops sharing a coordinate with another "
            "unmatched stop; retained but excluded from default map."
        ),
        "unique_unmatched_provisional": (
            "Unique unmatched GTFS coordinate; retained as "
            "provisional map coordinate."
        )
    },
    "limitations": [
        "GTFS and GeoJSON stop coverage is not complete.",
        "Provisional coordinates have not been independently verified.",
        "Repeated coordinates are flagged for review, not declared invalid.",
        "This is static geographic reference data, not live vehicle tracking."
    ]
}

with open(
    output_dir / "stop_dataset_metadata.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(metadata, file, indent=4, ensure_ascii=False)

# Display export results
print("Export validation passed.")
print("\nFiles created:")

for file_path in sorted(output_dir.iterdir()):
    if file_path.is_file():
        print(
            f"{file_path.name}: "
            f"{file_path.stat().st_size / 1024:.2f} KB"
        )

print("\nQuality summary:")
display(quality_summary)

print("\nMap dataset preview:")
display(stops_for_map.head(10))

Export validation passed.

Files created:
prepared_stops.csv: 1087.75 KB
stop_dataset_metadata.json: 1.10 KB
stop_quality_summary.csv: 0.18 KB
stops_for_map.csv: 940.90 KB

Quality summary:


,coordinate_status,stop_count,map_included
0,placeholder_candidate,1634,0
1,repeated_coordinate_review,75,0
2,unique_unmatched_provisional,1210,1210
3,verified_geojson_match,4209,4209



Map dataset preview:


,stop_id,stop_name,stop_id_match,stop_id_geo,stop_name_geo,stop_lat,stop_lon,latitude,longitude,map_lat,map_lon,distance_metres,coordinate_group_size,coordinate_status,include_in_default_map
18,fa11bdb91361aafff55ff4db66c424d2,BROADWAY,fa11bdb91361aafff55ff4db66c424d2,NaN,NaN,13.08703,80.28440,NaN,NaN,13.087030,80.284400,NaN,1,unique_unmatched_provisional,True
19,cGoHBmgo,HIGH COURT,cGoHBmgo,MTC_cGoHBmgo,High Court,13.08858,80.28712,13.088636,80.287111,13.088636,80.287111,6.335008,1,verified_geojson_match,True
20,fwfbiPwk,PARRYS CORNER,fwfbiPwk,MTC_fwfbiPwk,Parrys Corner,13.08869,80.29045,13.088694,80.290431,13.088694,80.290431,2.080166,1,verified_geojson_match,True
21,mgTKojDA,R B I PARRYS,mgTKojDA,MTC_mgTKojDA,R B I Parrys,13.08238,80.28898,13.082351,80.289052,13.082351,80.289052,8.411081,1,verified_geojson_match,True
22,drmArhka,SECRETARIAT,drmArhka,MTC_drmArhka,Secretariat,13.08002,80.28820,13.080017,80.288254,13.080017,80.288254,5.816329,1,verified_geojson_match,True
23,XypZkLUe,WAR MEMORIAL,XypZkLUe,MTC_XypZkLUe,War Memorial,13.07422,80.28619,13.074203,80.286248,13.074203,80.286248,6.600778,1,verified_geojson_match,True
24,gaLZwYvS,ANNA SQUARE B.T,gaLZwYvS,MTC_gaLZwYvS,Anna Square,13.06642,80.28431,13.066390,80.284400,13.066390,80.284400,10.334471,1,verified_geojson_match,True
25,qFzqwYZZ,CHEPPAKAM,qFzqwYZZ,MTC_qFzqwYZZ,Marina Beach,13.06130,80.28285,13.061291,80.282887,13.061291,80.282887,4.126279,1,verified_geojson_match,True
26,MBTcSIip,KANNAGI STATUE OR PRESIDENCY COLLEGE,MBTcSIip,MTC_MBTcSIip,Kannagi Statue Or Presidency College,13.05680,80.28191,13.056792,80.281954,13.056792,80.281954,4.882739,1,verified_geojson_match,True
27,ppabfQGz,VIVEKANANDA HOUSE BUS TERMINUS,ppabfQGz,MTC_ppabfQGz,Swami Vivekananda House,13.05038,80.28082,13.050370,80.280883,13.050370,80.280883,6.908146,1,verified_geojson_match,True


In [28]:
print("DataFrame type:", type(geo_route_name_df))
print("\nActual columns:")
print(geo_route_name_df.columns.tolist())

print("\nFirst 5 rows:")
display(geo_route_name_df.head())

print("\nDataFrame shape:")
print(geo_route_name_df.shape)

DataFrame type: <class 'pandas.core.frame.DataFrame'>

Actual columns:
['type', 'properties.route_name', 'properties.source', 'properties.destinatio', 'geometry.type', 'geometry.coordinates']

First 5 rows:


,type,properties.route_name,properties.source,properties.destinatio,geometry.type,geometry.coordinates
0,Feature,1,Thiruvottiyur Bus Stand,Thiruvanmiyur Depot,MultiLineString,"[[[80.30608, 13.17511], [80.30621, 13.17562], ..."
1,Feature,1B,Thiruvottiyur Bus Stand,Tambaram West Bus Stand,MultiLineString,"[[[80.30619, 13.1751], [80.30574, 13.17346], [..."
2,Feature,1C,Thiruvanmiyur Depot,Ennore Bus Depot,MultiLineString,"[[[80.25932, 12.98733], [80.25736, 12.98744], ..."
3,Feature,1D,Thiruvanmiyur Depot,Ennore Bus Depot,MultiLineString,"[[[80.25931, 12.98731], [80.25803, 12.98741], ..."
4,Feature,2,Anna Square,Kaviarasu Kannadasan Nagar Bus Stand,MultiLineString,"[[[80.28431, 13.06642], [80.28423, 13.06628], ..."



DataFrame shape:
(637, 6)


In [30]:
# Step 19A: Route reconciliation inventory (corrected)

# 1. Create working copies
gtfs_routes_check = gtfs_data["routes"].copy()

geo_routes_check = geo_route_name_df[
    [
        "properties.route_name",
        "properties.source",
        "properties.destinatio"
    ]
].copy()

# 2. Rename GeoJSON columns for easier processing
geo_routes_check = geo_routes_check.rename(columns={
    "properties.route_name": "route_name",
    "properties.source": "source",
    "properties.destinatio": "destination"
})

# 3. Normalise route names and identifiers
gtfs_routes_check["route_short_norm"] = (
    gtfs_routes_check["route_short_name"]
    .fillna("")
    .apply(normalize_name)
)

gtfs_routes_check["route_long_norm"] = (
    gtfs_routes_check["route_long_name"]
    .fillna("")
    .apply(normalize_name)
)

geo_routes_check["route_name_norm"] = (
    geo_routes_check["route_name"]
    .fillna("")
    .apply(normalize_name)
)

geo_routes_check["source_norm"] = (
    geo_routes_check["source"]
    .fillna("")
    .apply(normalize_name)
)

geo_routes_check["destination_norm"] = (
    geo_routes_check["destination"]
    .fillna("")
    .apply(normalize_name)
)

# 4. Compare GTFS route numbers with GeoJSON route names
gtfs_route_names = set(
    gtfs_routes_check["route_short_norm"].dropna()
) - {""}

geo_route_names = set(
    geo_routes_check["route_name_norm"].dropna()
) - {""}

common_route_names = gtfs_route_names.intersection(geo_route_names)

print("GTFS route records:", len(gtfs_routes_check))
print("GTFS unique route IDs:", gtfs_routes_check["route_id"].nunique())
print("GTFS unique route numbers:", len(gtfs_route_names))

print("\nGeoJSON route features:", len(geo_routes_check))
print("GeoJSON unique route names:", len(geo_route_names))

print("\nExact route-number/name matches:", len(common_route_names))

# 5. Display sample matching routes
print("\nSample matching route numbers:")
display(
    geo_routes_check[
        geo_routes_check["route_name_norm"].isin(common_route_names)
    ][["route_name", "source", "destination"]]
    .head(15)
)

# 6. Inspect selected GTFS route-number variants

selected_route_numbers = ["1", "101", "102", "102A", "102P"]

print("\nSelected GTFS route-number variants:")

display(
    gtfs_routes_check[
        gtfs_routes_check["route_short_norm"].isin(selected_route_numbers)
    ][
        ["route_id", "route_short_name", "route_long_name"]
    ].sort_values(
        ["route_short_name", "route_id"]
    ).head(100)
)

# 7. Inspect corresponding GeoJSON route features

print("\nCorresponding GeoJSON route features:")

display(
    geo_routes_check[
        geo_routes_check["route_name_norm"].isin(selected_route_numbers)
    ][["route_name", "source", "destination"]]
)

GTFS route records: 4030
GTFS unique route IDs: 4030
GTFS unique route numbers: 998

GeoJSON route features: 637
GeoJSON unique route names: 637

Exact route-number/name matches: 595

Sample matching route numbers:


,route_name,source,destination
0,1,Thiruvottiyur Bus Stand,Thiruvanmiyur Depot
1,1B,Thiruvottiyur Bus Stand,Tambaram West Bus Stand
2,1C,Thiruvanmiyur Depot,Ennore Bus Depot
3,1D,Thiruvanmiyur Depot,Ennore Bus Depot
5,2A,Kaviarasu Kannadasan Nagar Bus Stand,Anna Square
6,2ACT,Anna Square,MGR Central Station
7,3,Thiyagaraya Nagar Bus Depot,Thiruvanmiyur Depot
8,3CT,Tidel Park,Guindy Railway Station
9,4,Broadway,Ennore Bus Depot
10,4M,Broadway,Minjur Terminus



Selected GTFS route-number variants:


,route_id,route_short_name,route_long_name
1360,2561,1,THIRUVOTRIYUR B.T To BROADWAY
1361,2562,1,BROADWAY To THIRUVOTRIYUR B.T
1362,2563,1,THIRUVOTRIYUR B.T To TONDIARPET
1363,2564,1,TONDIARPET To THIRUVOTRIYUR B.T
510,966,1,THIRUVOTRIYUR B.T To THIRUVANMIYUR
511,967,1,M.G.R.CENTRAL To THIRUVANMIYUR
515,986,1,THIRUVANMIYUR To M.G.R.CENTRAL
516,987,1,THIRUVANMIYUR To THIRUVOTRIYUR B.T
827,1933,101,BROADWAY To POONAMALLEE B.T
828,1934,101,POONAMALLEE B.T To BROADWAY



Corresponding GeoJSON route features:


,route_name,source,destination
0,1,Thiruvottiyur Bus Stand,Thiruvanmiyur Depot
340,101,Poonamallee Terminus,Thiruvottiyur Bus Stand
342,102,Broadway,Kelambakkam Bus Terminal
343,102A,Pudupakkam Or Vandaloor Road Junction,Thiruvanmiyur Depot
347,102P,Broadway,Perumbakkam SCB


In [31]:
# Step 19B: Route number + origin-destination reconciliation

# 1. Prepare GTFS route records
gtfs_od_check = gtfs_routes_check.copy()

# Split route_long_name into origin and destination using "To"
gtfs_od_parts = (
    gtfs_od_check["route_long_name"]
    .fillna("")
    .str.split(r"\s+TO\s+", n=1, regex=True, expand=True)
)

gtfs_od_check["gtfs_origin"] = gtfs_od_parts[0]
gtfs_od_check["gtfs_destination"] = (
    gtfs_od_parts[1] if 1 in gtfs_od_parts.columns else ""
)

# Normalise endpoints
gtfs_od_check["origin_norm"] = (
    gtfs_od_check["gtfs_origin"].apply(normalize_name)
)

gtfs_od_check["destination_norm"] = (
    gtfs_od_check["gtfs_destination"].apply(normalize_name)
)

# 2. Prepare GeoJSON route records
geo_od_check = geo_routes_check.copy()

# 3. Count records where origin/destination could be extracted
valid_gtfs_od = (
    (gtfs_od_check["origin_norm"] != "") &
    (gtfs_od_check["destination_norm"] != "")
)

print("GTFS route records:", len(gtfs_od_check))
print("GTFS records with extractable origin/destination:",
      int(valid_gtfs_od.sum()))
print("GTFS records without extractable endpoints:",
      int((~valid_gtfs_od).sum()))

# 4. Match in the same direction
forward_matches = gtfs_od_check.merge(
    geo_od_check,
    left_on=[
        "route_short_norm",
        "origin_norm",
        "destination_norm"
    ],
    right_on=[
        "route_name_norm",
        "source_norm",
        "destination_norm"
    ],
    how="inner",
    suffixes=("_gtfs", "_geo")
)

# 5. Match in the reverse direction
reverse_matches = gtfs_od_check.merge(
    geo_od_check,
    left_on=[
        "route_short_norm",
        "origin_norm",
        "destination_norm"
    ],
    right_on=[
        "route_name_norm",
        "destination_norm",
        "source_norm"
    ],
    how="inner",
    suffixes=("_gtfs", "_geo")
)

# 6. Summarise matching results
print("\n--- Route reconciliation results ---")

print("Same-direction matching records:",
      len(forward_matches))

print("Same-direction unique GTFS route IDs:",
      forward_matches["route_id"].nunique())

print("Reverse-direction matching records:",
      len(reverse_matches))

print("Reverse-direction unique GTFS route IDs:",
      reverse_matches["route_id"].nunique())

print("\nSample same-direction matches:")

display(
    forward_matches[
        [
            "route_id",
            "route_short_name",
            "route_long_name",
            "route_name",
            "source",
            "destination"
        ]
    ].head(20)
)

print("\nSample reverse-direction matches:")

display(
    reverse_matches[
        [
            "route_id",
            "route_short_name",
            "route_long_name",
            "route_name",
            "source",
            "destination"
        ]
    ].head(20)
)

GTFS route records: 4030
GTFS records with extractable origin/destination: 0
GTFS records without extractable endpoints: 4030

--- Route reconciliation results ---
Same-direction matching records: 0
Same-direction unique GTFS route IDs: 0
Reverse-direction matching records: 0
Reverse-direction unique GTFS route IDs: 0

Sample same-direction matches:


,route_id,route_short_name,route_long_name,route_name,source,destination



Sample reverse-direction matches:


,route_id,route_short_name,route_long_name,route_name,source,destination


In [33]:
# Step 19B: Route number + origin-destination reconciliation

import re

gtfs_od_check = gtfs_routes_check.copy()
geo_od_check = geo_routes_check.copy()

# 1. Split GTFS route names using case-insensitive "TO"
def split_route_name(route_name):
    if pd.isna(route_name):
        return "", ""

    parts = re.split(
        r"\s+TO\s+",
        str(route_name),
        maxsplit=1,
        flags=re.IGNORECASE
    )

    if len(parts) == 2:
        return parts[0].strip(), parts[1].strip()

    return "", ""

gtfs_od_check[["gtfs_origin", "gtfs_destination"]] = (
    gtfs_od_check["route_long_name"]
    .apply(split_route_name)
    .apply(pd.Series)
)

# 2. Normalise endpoints
gtfs_od_check["origin_norm"] = (
    gtfs_od_check["gtfs_origin"].apply(normalize_name)
)

gtfs_od_check["destination_norm"] = (
    gtfs_od_check["gtfs_destination"].apply(normalize_name)
)

# 3. Check extraction quality
valid_gtfs_od = (
    (gtfs_od_check["origin_norm"] != "") &
    (gtfs_od_check["destination_norm"] != "")
)

print("GTFS route records:", len(gtfs_od_check))
print("Records with extractable endpoints:", int(valid_gtfs_od.sum()))
print("Records without extractable endpoints:", int((~valid_gtfs_od).sum()))

print("\nSample extracted GTFS endpoints:")

display(
    gtfs_od_check[
        ["route_short_name", "route_long_name",
         "gtfs_origin", "gtfs_destination"]
    ].head(15)
)

# 4. Same-direction matching
forward_matches = gtfs_od_check.merge(
    geo_od_check,
    left_on=["route_short_norm", "origin_norm", "destination_norm"],
    right_on=["route_name_norm", "source_norm", "destination_norm"],
    how="inner",
    suffixes=("_gtfs", "_geo")
)

# 5. Reverse-direction matching
reverse_matches = gtfs_od_check.merge(
    geo_od_check,
    left_on=["route_short_norm", "origin_norm", "destination_norm"],
    right_on=["route_name_norm", "destination_norm", "source_norm"],
    how="inner",
    suffixes=("_gtfs", "_geo")
)

# 6. Report results
print("\n--- Route reconciliation results ---")

print("Same-direction matching records:", len(forward_matches))
print("Same-direction unique GTFS route IDs:",
      forward_matches["route_id"].nunique())

print("Reverse-direction matching records:", len(reverse_matches))
print("Reverse-direction unique GTFS route IDs:",
      reverse_matches["route_id"].nunique())

print("\nSample same-direction matches:")

display(
    forward_matches[
        ["route_id", "route_short_name", "route_long_name",
         "route_name", "source", "destination"]
    ].head(20)
)

print("\nSample reverse-direction matches:")

display(
    reverse_matches[
        ["route_id", "route_short_name", "route_long_name",
         "route_name", "source", "destination"]
    ].head(20)
)

GTFS route records: 4030
Records with extractable endpoints: 4030
Records without extractable endpoints: 0

Sample extracted GTFS endpoints:


,route_short_name,route_long_name,gtfs_origin,gtfs_destination
0,102,ADYAR B.T To SIRUSERI I.T PARK,ADYAR B.T,SIRUSERI I.T PARK
1,102,BROADWAY To KELAMBAKKAM B.T,BROADWAY,KELAMBAKKAM B.T
2,102,KELAMBAKKAM B.T To BROADWAY,KELAMBAKKAM B.T,BROADWAY
3,102,BROADWAY To SIRUSERI I.T.PARK,BROADWAY,SIRUSERI I.T.PARK
4,102,SIRUSERI I.T.PARK To BROADWAY,SIRUSERI I.T.PARK,BROADWAY
5,102,M.G.R.CENTRAL To KELAMBAKKAM,M.G.R.CENTRAL,KELAMBAKKAM
6,102C,BROADWAY To SEMMANCHERI BUS STAND,BROADWAY,SEMMANCHERI BUS STAND
7,102C,SEMMANCHERI BUS STAND To BROADWAY,SEMMANCHERI BUS STAND,BROADWAY
8,102C,TIDEL PARK To SEMMANCHERI TNUHDB,TIDEL PARK,SEMMANCHERI TNUHDB
9,102C,SEMMANCHERI TNUHDB To TIDEL PARK,SEMMANCHERI TNUHDB,TIDEL PARK



--- Route reconciliation results ---
Same-direction matching records: 101
Same-direction unique GTFS route IDs: 101
Reverse-direction matching records: 100
Reverse-direction unique GTFS route IDs: 100

Sample same-direction matches:


,route_id,route_short_name,route_long_name,route_name,source,destination
0,17,102C,BROADWAY To SEMMANCHERI BUS STAND,102C,Broadway,Semmancheri Bus Stand
1,46,102X,THIRUPORUR BUS STAND To BROADWAY,102X,Thiruporur Bus Stand,Broadway
2,88,15F,VADAPALANI BUS STAND To BROADWAY,15F,Vadapalani Bus Stand,Broadway
3,91,17D,BROADWAY To K.K.NAGAR,17D,Broadway,K K Nagar
4,96,17E,SALIGRAMAM BUS STAND To BROADWAY,17E,Saligramam Bus Stand,Broadway
5,176,50ET,AYYAPPAN MADAM To BROADWAY,50ET,Ayyappan Madam,Broadway
6,239,A51,BROADWAY To TAMBARAM WEST BUS STAND,A51,Broadway,Tambaram West Bus Stand
7,246,E18,GUDUVANCHERI BUS TERMINUS To BROADWAY,E18,Guduvancheri Bus Terminus,Broadway
8,317,S5,FORT RAILWAY STATION To SECRETARIAT,S5,Fort Railway Station,Secretariat
9,509,51,VELACHERY To TAMBARAM WEST BUS STAND,51,Velachery,Tambaram West Bus Stand



Sample reverse-direction matches:


,route_id,route_short_name,route_long_name,route_name,source,destination
0,18,102C,SEMMANCHERI BUS STAND To BROADWAY,102C,Broadway,Semmancheri Bus Stand
1,45,102X,BROADWAY To THIRUPORUR BUS STAND,102X,Thiruporur Bus Stand,Broadway
2,79,109X,BROADWAY To THIRUPORUR BUS STAND,109X,Thiruporur Bus Stand,Broadway
3,87,15F,BROADWAY To VADAPALANI BUS STAND,15F,Vadapalani Bus Stand,Broadway
4,92,17D,K.K.NAGAR To BROADWAY,17D,Broadway,K K Nagar
5,95,17E,BROADWAY To SALIGRAMAM BUS STAND,17E,Saligramam Bus Stand,Broadway
6,175,50ET,BROADWAY To AYYAPPAN MADAM,50ET,Ayyappan Madam,Broadway
7,240,A51,TAMBARAM WEST BUS STAND To BROADWAY,A51,Broadway,Tambaram West Bus Stand
8,325,S5,SECRETARIAT To FORT RAILWAY STATION,S5,Fort Railway Station,Secretariat
9,505,51,TAMBARAM WEST BUS STAND To VELACHERY,51,Velachery,Tambaram West Bus Stand


In [34]:
# Step 19C: Route mapping uniqueness audit

# 1. Collect matched GeoJSON route names
forward_geo_names = set(
    forward_matches["route_name_norm"].dropna()
)

reverse_geo_names = set(
    reverse_matches["route_name_norm"].dropna()
)

all_matched_geo_names = forward_geo_names.union(reverse_geo_names)

# 2. Combine directional matches
all_route_matches = pd.concat(
    [
        forward_matches.assign(match_direction="same_direction"),
        reverse_matches.assign(match_direction="reverse_direction")
    ],
    ignore_index=True
)

# 3. Count unique matches
print("Forward matched GeoJSON features:",
      len(forward_geo_names))

print("Reverse matched GeoJSON features:",
      len(reverse_geo_names))

print("Distinct GeoJSON features matched overall:",
      len(all_matched_geo_names))

print("Combined matching records:",
      len(all_route_matches))

print("Unique GTFS route IDs matched overall:",
      all_route_matches["route_id"].nunique())

# 4. Check whether one GTFS route matches multiple GeoJSON features
gtfs_mapping_counts = (
    all_route_matches.groupby("route_id")["route_name_norm"]
    .nunique()
)

print("\nGTFS route IDs linked to multiple GeoJSON features:",
      int((gtfs_mapping_counts > 1).sum()))

# 5. Check whether one GeoJSON feature matches multiple GTFS route IDs
geo_mapping_counts = (
    all_route_matches.groupby("route_name_norm")["route_id"]
    .nunique()
)

print("GeoJSON features linked to multiple GTFS route IDs:",
      int((geo_mapping_counts > 1).sum()))

# 6. Display ambiguous GTFS mappings
print("\nGTFS routes with multiple GeoJSON feature mappings:")

display(
    gtfs_mapping_counts[gtfs_mapping_counts > 1]
    .rename("geojson_feature_count")
    .reset_index()
)

# 7. Identify unmatched GeoJSON features
unmatched_geo_routes = geo_od_check[
    ~geo_od_check["route_name_norm"].isin(all_matched_geo_names)
].copy()

print("\nUnmatched GeoJSON route features:",
      len(unmatched_geo_routes))

print("\nSample unmatched GeoJSON routes:")

display(
    unmatched_geo_routes[
        ["route_name", "source", "destination"]
    ].head(20)
)

Forward matched GeoJSON features: 101
Reverse matched GeoJSON features: 100
Distinct GeoJSON features matched overall: 120
Combined matching records: 201
Unique GTFS route IDs matched overall: 201

GTFS route IDs linked to multiple GeoJSON features: 0
GeoJSON features linked to multiple GTFS route IDs: 81

GTFS routes with multiple GeoJSON feature mappings:


,route_id,geojson_feature_count



Unmatched GeoJSON route features: 517

Sample unmatched GeoJSON routes:


,route_name,source,destination
0,1,Thiruvottiyur Bus Stand,Thiruvanmiyur Depot
1,1B,Thiruvottiyur Bus Stand,Tambaram West Bus Stand
2,1C,Thiruvanmiyur Depot,Ennore Bus Depot
3,1D,Thiruvanmiyur Depot,Ennore Bus Depot
4,2,Anna Square,Kaviarasu Kannadasan Nagar Bus Stand
5,2A,Kaviarasu Kannadasan Nagar Bus Stand,Anna Square
6,2ACT,Anna Square,MGR Central Station
7,3,Thiyagaraya Nagar Bus Depot,Thiruvanmiyur Depot
9,4,Broadway,Ennore Bus Depot
10,4M,Broadway,Minjur Terminus


In [35]:
# Step 19D: Inspect GeoJSON features linked to multiple GTFS routes

# 1. Identify GeoJSON features with multiple GTFS route IDs
ambiguous_geo_names = (
    geo_mapping_counts[geo_mapping_counts > 1].index
)

ambiguous_geo_matches = all_route_matches[
    all_route_matches["route_name_norm"].isin(ambiguous_geo_names)
].copy()

# 2. Display summary
print("Number of ambiguous GeoJSON features:",
      len(ambiguous_geo_names))

print("GTFS route records involved:",
      len(ambiguous_geo_matches))

print("Unique GTFS route IDs involved:",
      ambiguous_geo_matches["route_id"].nunique())

# 3. Group matching records by GeoJSON feature
ambiguous_geo_summary = (
    ambiguous_geo_matches
    .groupby("route_name_norm")
    .agg(
        route_name=("route_name", "first"),
        source=("source", "first"),
        destination=("destination", "first"),
        gtfs_route_ids=("route_id", lambda x: list(x)),
        gtfs_route_names=("route_long_name", lambda x: list(x)),
        match_directions=("match_direction", lambda x: list(x))
    )
    .reset_index()
)

# 4. Display first 30 ambiguous mappings
display(ambiguous_geo_summary.head(30))

# 5. Check how many GTFS route records share identical route numbers
# and endpoint combinations
duplicate_gtfs_combinations = (
    gtfs_od_check[
        ["route_short_norm", "origin_norm", "destination_norm"]
    ]
    .value_counts()
    .reset_index(name="record_count")
)

print("\nGTFS route-number and endpoint combinations occurring more than once:")

display(
    duplicate_gtfs_combinations[
        duplicate_gtfs_combinations["record_count"] > 1
    ].head(30)
)

Number of ambiguous GeoJSON features: 81
GTFS route records involved: 162
Unique GTFS route IDs involved: 162


,route_name_norm,route_name,source,destination,gtfs_route_ids,gtfs_route_names,match_directions
0,102C,102C,Broadway,Semmancheri Bus Stand,"[17, 18]","[BROADWAY To SEMMANCHERI BUS STAND, SEMMANCHER...","[same_direction, reverse_direction]"
1,102X,102X,Thiruporur Bus Stand,Broadway,"[46, 45]","[THIRUPORUR BUS STAND To BROADWAY, BROADWAY To...","[same_direction, reverse_direction]"
2,118,118,Tambaram West Bus Stand,Maraimalai Nagar Bus Stand,"[2138, 2139]",[TAMBARAM WEST BUS STAND To MARAIMALAI NAGAR B...,"[same_direction, reverse_direction]"
3,118ET,118ET,Kalivanthapattu,Tambaram West Bus Stand,"[2143, 2142]","[KALIVANTHAPATTU To TAMBARAM WEST BUS STAND, T...","[same_direction, reverse_direction]"
4,118N,118N,Tambaram West Bus Stand,Neelamangalam,"[2146, 2147]","[TAMBARAM WEST BUS STAND To NEELAMANGALAM, NEE...","[same_direction, reverse_direction]"
5,11G,11G,K K Nagar,Broadway,"[2161, 2160]","[K.K.NAGAR To BROADWAY, BROADWAY To K.K.NAGAR]","[same_direction, reverse_direction]"
6,120E,120E,Kilkondaiyur,Broadway,"[2192, 2191]","[KILKONDAIYUR To BROADWAY, BROADWAY To KILKOND...","[same_direction, reverse_direction]"
7,120G,120G,Menambedu,Broadway,"[2204, 2203]","[MENAMBEDU To BROADWAY, BROADWAY To MENAMBEDU]","[same_direction, reverse_direction]"
8,120K,120K,Broadway,Kadhavoor Bus Stand,"[2209, 2210]","[BROADWAY To KADHAVOOR BUS STAND, KADHAVOOR BU...","[same_direction, reverse_direction]"
9,120KET,120KET,Kilkondaiyur,Broadway,"[2214, 2213]","[KILKONDAIYUR To BROADWAY, BROADWAY To KILKOND...","[same_direction, reverse_direction]"



GTFS route-number and endpoint combinations occurring more than once:


,route_short_norm,origin_norm,destination_norm,record_count
0,102PCT,BROADWAY,PERUMBAKKAM TNUHDB,2
1,56P,TONDIARPET,MINJUR NEITHAVOYAL,2
2,57B,POTHUR,REDHILLS,2
3,57B,REDHILLS,POTHUR,2
4,57CT,PADIYANALLUR,REDHILLS,2
5,57CT,REDHILLS,PADIYANALLUR,2
6,63,THIRUVERKADU,I C F,2
7,70C,KILAMBAKKAM BUS TERMINUS,M G R KOYAMBEDU B T,2
8,A51CT,VELACHERY B T,BROADWAY,2
9,M51V,KOLATHUR,SAIDAPET,2


In [36]:
# Step 19E: Inspect GeoJSON route geometry endpoints

selected_routes = ["102C", "102X", "118", "15F", "17D"]

for route_number in selected_routes:

    route_rows = geo_route_name_df[
        geo_route_name_df["properties.route_name"]
        .astype(str)
        .str.strip()
        .str.upper()
        == route_number
    ]

    print(f"\n{'=' * 60}")
    print(f"ROUTE: {route_number}")
    print(f"{'=' * 60}")

    if route_rows.empty:
        print("No GeoJSON feature found.")
        continue

    for _, route_row in route_rows.iterrows():

        coordinates = route_row["geometry.coordinates"]

        print("Source:", route_row["properties.source"])
        print("Destination:", route_row["properties.destinatio"])
        print("Geometry type:", route_row["geometry.type"])
        print("Number of line components:", len(coordinates))

        for component_number, line in enumerate(coordinates, start=1):

            if not line:
                continue

            start_point = line[0]
            end_point = line[-1]

            print(f"\nComponent {component_number}")
            print("Coordinate count:", len(line))
            print("Start [longitude, latitude]:", start_point)
            print("End   [longitude, latitude]:", end_point)


ROUTE: 102C
Source: Broadway
Destination: Semmancheri Bus Stand
Geometry type: MultiLineString
Number of line components: 1

Component 1
Coordinate count: 458
Start [longitude, latitude]: [80.28382, 13.08675]
End   [longitude, latitude]: [80.21007, 12.87458]

ROUTE: 102X
Source: Thiruporur Bus Stand
Destination: Broadway
Geometry type: MultiLineString
Number of line components: 1

Component 1
Coordinate count: 561
Start [longitude, latitude]: [80.18728, 12.72213]
End   [longitude, latitude]: [80.28367, 13.08785]

ROUTE: 118
Source: Tambaram West Bus Stand
Destination: Maraimalai Nagar Bus Stand
Geometry type: MultiLineString
Number of line components: 1

Component 1
Coordinate count: 237
Start [longitude, latitude]: [80.11938, 12.92836]
End   [longitude, latitude]: [80.03263, 12.79108]

ROUTE: 15F
Source: Vadapalani Bus Stand
Destination: Broadway
Geometry type: MultiLineString
Number of line components: 1

Component 1
Coordinate count: 274
Start [longitude, latitude]: [80.20716, 13.0

In [38]:
import numpy as np
import pandas as pd

# Corrected Haversine function
def haversine_m(lat1, lon1, lat2, lon2):
    earth_radius = 6371000

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    return 2 * earth_radius * np.arcsin(
        np.sqrt(np.clip(a, 0, 1))
    )


# Use stops with valid selected map coordinates
valid_stops = prepared_stops.dropna(
    subset=["map_lat", "map_lon"]
).copy()

# Ensure coordinate columns are numeric
valid_stops["map_lat"] = pd.to_numeric(
    valid_stops["map_lat"], errors="coerce"
)
valid_stops["map_lon"] = pd.to_numeric(
    valid_stops["map_lon"], errors="coerce"
)

valid_stops = valid_stops.dropna(
    subset=["map_lat", "map_lon"]
)

route_numbers = ["102C", "102X", "118", "15F", "17D"]

endpoint_results = []

for route_number in route_numbers:

    route_rows = geo_route_name_df[
        geo_route_name_df["properties.route_name"]
        .astype(str).str.strip().str.upper()
        == route_number
    ]

    if route_rows.empty:
        continue

    route_row = route_rows.iloc[0]
    line = route_row["geometry.coordinates"][0]

    endpoints = {
        "start": line[0],
        "end": line[-1]
    }

    for endpoint_type, point in endpoints.items():

        lon, lat = point

        distances = haversine_m(
            lat,
            lon,
            valid_stops["map_lat"].to_numpy(dtype=float),
            valid_stops["map_lon"].to_numpy(dtype=float)
        )

        nearest_index = int(np.argmin(distances))
        nearest_stop = valid_stops.iloc[nearest_index]

        endpoint_results.append({
            "route_number": route_number,
            "endpoint": endpoint_type,
            "endpoint_lat": lat,
            "endpoint_lon": lon,
            "nearest_stop_id": nearest_stop["stop_id"],
            "nearest_stop_name": nearest_stop["stop_name"],
            "distance_m": round(float(distances[nearest_index]), 2)
        })


endpoint_check_df = pd.DataFrame(endpoint_results)

display(
    endpoint_check_df.sort_values(
        ["route_number", "endpoint"]
    ).reset_index(drop=True)
)

,route_number,endpoint,endpoint_lat,endpoint_lon,nearest_stop_id,nearest_stop_name,distance_m
0,102C,end,12.87458,80.21007,DFiOxLfP,SEMMANCHERI BUS STAND,11.47
1,102C,start,13.08675,80.28382,fa11bdb91361aafff55ff4db66c424d2,BROADWAY,70.11
2,102X,end,13.08785,80.28367,EkWBPXmr,BROADWAY,46.16
3,102X,start,12.72213,80.18728,mAREpQlu,THIRUPORUR BUS STAND,2.79
4,118,end,12.79108,80.03263,CkSrqRXW,MARAIMALAI NAGAR BUS STAND,4.25
5,118,start,12.92836,80.11938,60b1deb61b7391af4c7c2a2017479dd2,TAMBARAM WEST BUS STAND,82.02
6,15F,end,13.08836,80.28388,134ee62a,BROADWAY,3.34
7,15F,start,13.04900,80.20716,lqMPnIra,VADAPALANI BUS STAND,38.89
8,17D,end,13.03586,80.20541,SMhobJmy,K.K.NAGAR,6.99
9,17D,start,13.08836,80.28388,134ee62a,BROADWAY,3.34


In [39]:
print("all_route_matches shape:", all_route_matches.shape)
print("\nColumns:")
print(all_route_matches.columns.tolist())

display(all_route_matches.head(5))

all_route_matches shape: (201, 19)

Columns:
['route_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_type', 'route_short_norm', 'route_long_norm', 'gtfs_origin', 'gtfs_destination', 'origin_norm', 'destination_norm', 'route_name', 'source', 'destination', 'route_name_norm', 'source_norm', 'match_direction', 'destination_norm_gtfs', 'destination_norm_geo']


,route_id,agency_id,route_short_name,route_long_name,route_type,route_short_norm,route_long_norm,gtfs_origin,gtfs_destination,origin_norm,destination_norm,route_name,source,destination,route_name_norm,source_norm,match_direction,destination_norm_gtfs,destination_norm_geo
0,17,CUMTA,102C,BROADWAY To SEMMANCHERI BUS STAND,3,102C,BROADWAY TO SEMMANCHERI BUS STAND,BROADWAY,SEMMANCHERI BUS STAND,BROADWAY,SEMMANCHERI BUS STAND,102C,Broadway,Semmancheri Bus Stand,102C,BROADWAY,same_direction,NaN,NaN
1,46,CUMTA,102X,THIRUPORUR BUS STAND To BROADWAY,3,102X,THIRUPORUR BUS STAND TO BROADWAY,THIRUPORUR BUS STAND,BROADWAY,THIRUPORUR BUS STAND,BROADWAY,102X,Thiruporur Bus Stand,Broadway,102X,THIRUPORUR BUS STAND,same_direction,NaN,NaN
2,88,CUMTA,15F,VADAPALANI BUS STAND To BROADWAY,3,15F,VADAPALANI BUS STAND TO BROADWAY,VADAPALANI BUS STAND,BROADWAY,VADAPALANI BUS STAND,BROADWAY,15F,Vadapalani Bus Stand,Broadway,15F,VADAPALANI BUS STAND,same_direction,NaN,NaN
3,91,CUMTA,17D,BROADWAY To K.K.NAGAR,3,17D,BROADWAY TO K K NAGAR,BROADWAY,K.K.NAGAR,BROADWAY,K K NAGAR,17D,Broadway,K K Nagar,17D,BROADWAY,same_direction,NaN,NaN
4,96,CUMTA,17E,SALIGRAMAM BUS STAND To BROADWAY,3,17E,SALIGRAMAM BUS STAND TO BROADWAY,SALIGRAMAM BUS STAND,BROADWAY,SALIGRAMAM BUS STAND,BROADWAY,17E,Saligramam Bus Stand,Broadway,17E,SALIGRAMAM BUS STAND,same_direction,NaN,NaN


In [40]:
import numpy as np
import pandas as pd

# Haversine distance in metres
def haversine_m(lat1, lon1, lat2, lon2):
    earth_radius = 6371000

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    return 2 * earth_radius * np.arcsin(
        np.sqrt(np.clip(a, 0, 1))
    )


# Prepare valid map stops
valid_stops = prepared_stops.dropna(
    subset=["map_lat", "map_lon"]
).copy()

valid_stops["map_lat"] = pd.to_numeric(
    valid_stops["map_lat"], errors="coerce"
)
valid_stops["map_lon"] = pd.to_numeric(
    valid_stops["map_lon"], errors="coerce"
)

valid_stops = valid_stops.dropna(
    subset=["map_lat", "map_lon"]
)

stop_lats = valid_stops["map_lat"].to_numpy(dtype=float)
stop_lons = valid_stops["map_lon"].to_numpy(dtype=float)


# Get each distinct GeoJSON route feature only once
unique_routes = all_route_matches[
    ["route_name", "source", "destination"]
].drop_duplicates(subset=["route_name"]).copy()

endpoint_results = []

for _, route in unique_routes.iterrows():

    route_number = str(route["route_name"]).strip()

    geo_rows = geo_route_name_df[
        geo_route_name_df["properties.route_name"]
        .astype(str).str.strip().str.upper()
        == route_number.upper()
    ]

    if geo_rows.empty:
        continue

    geo_row = geo_rows.iloc[0]
    geometry_type = geo_row["geometry.type"]
    coordinates = geo_row["geometry.coordinates"]

    # Handle LineString and MultiLineString
    if geometry_type == "MultiLineString":
        components = coordinates
    elif geometry_type == "LineString":
        components = [coordinates]
    else:
        continue

    if not components:
        continue

    # Route start = first coordinate of first component
    # Route end = last coordinate of last component
    endpoints = {
        "start": components[0][0],
        "end": components[-1][-1]
    }

    for endpoint_type, point in endpoints.items():

        lon, lat = point

        distances = haversine_m(
            lat, lon,
            stop_lats, stop_lons
        )

        nearest_index = int(np.argmin(distances))
        nearest_stop = valid_stops.iloc[nearest_index]

        endpoint_results.append({
            "route_number": route_number,
            "source": route["source"],
            "destination": route["destination"],
            "endpoint": endpoint_type,
            "nearest_stop_name": nearest_stop["stop_name"],
            "distance_m": round(
                float(distances[nearest_index]), 2
            ),
            "component_count": len(components),
            "geometry_type": geometry_type
        })


# Create endpoint validation table
route_endpoint_validation = pd.DataFrame(endpoint_results)

print("Distinct matched GeoJSON routes:", len(unique_routes))
print("Routes processed:", route_endpoint_validation["route_number"].nunique())
print("Endpoint records:", len(route_endpoint_validation))

# Endpoint-level summary
print("\nENDPOINT DISTANCE SUMMARY")

display(
    route_endpoint_validation["distance_m"].describe().round(2)
)

print("\nENDPOINT THRESHOLD COUNTS")

for threshold in [50, 100, 250]:
    passed = (
        route_endpoint_validation["distance_m"] <= threshold
    ).sum()

    total = len(route_endpoint_validation)

    print(
        f"Within {threshold} metres: "
        f"{passed}/{total} ({passed / total * 100:.1f}%)"
    )


# Route-level summary: worst endpoint distance per route
route_quality_summary = (
    route_endpoint_validation
    .groupby(
        ["route_number", "source", "destination"],
        as_index=False
    )
    .agg(
        start_distance_m=(
            "distance_m",
            lambda x: x.iloc[0]
        ),
        worst_endpoint_distance_m=("distance_m", "max"),
        component_count=("component_count", "max")
    )
    .sort_values(
        "worst_endpoint_distance_m",
        ascending=False
    )
)

print("\n20 ROUTES WITH THE LARGEST ENDPOINT DISTANCES")

display(route_quality_summary.head(20))

print("\nMULTI-COMPONENT ROUTE COUNT")

print(
    route_quality_summary[
        "component_count"
    ].gt(1).sum()
)

Distinct matched GeoJSON routes: 120
Routes processed: 120
Endpoint records: 240

ENDPOINT DISTANCE SUMMARY


count     240.00
mean       31.76
std       156.75
min         0.00
25%         2.65
50%         4.26
75%        11.07
max      1609.98
Name: distance_m, dtype: float64


ENDPOINT THRESHOLD COUNTS
Within 50 metres: 212/240 (88.3%)
Within 100 metres: 234/240 (97.5%)
Within 250 metres: 236/240 (98.3%)

20 ROUTES WITH THE LARGEST ENDPOINT DISTANCES


,route_number,source,destination,start_distance_m,worst_endpoint_distance_m,component_count
14,120KET,Kilkondaiyur,Broadway,1609.98,1609.98,1
11,120E,Kilkondaiyur,Broadway,1609.98,1609.98,1
113,S54TCT,Porur,Chembarambakkam,4.24,806.20,1
7,118N,Tambaram West Bus Stand,Neelamangalam,333.41,333.41,1
101,S12,Velachery Railway Station,Guindy Railway Station,144.03,144.03,1
66,555M,Tambaram West Bus Stand,Thiruporur Bus Stand,104.95,104.95,1
49,50ET,Ayyappan Madam,Broadway,4.59,82.28,1
4,118,Tambaram West Bus Stand,Maraimalai Nagar Bus Stand,82.02,82.02,1
12,120G,Menambedu,Broadway,4.20,81.93,1
93,E18,Guduvancheri Bus Terminus,Broadway,10.28,79.33,1



MULTI-COMPONENT ROUTE COUNT
0


In [41]:
# Identify routes with at least one endpoint over 100 metres
outlier_route_numbers = route_quality_summary.loc[
    route_quality_summary["worst_endpoint_distance_m"] > 100,
    "route_number"
].tolist()

# Show both endpoint records for those routes
outlier_details = route_endpoint_validation[
    route_endpoint_validation["route_number"].isin(
        outlier_route_numbers
    )
].copy()

outlier_details = outlier_details.sort_values(
    ["route_number", "endpoint"]
)

display(outlier_details.reset_index(drop=True))

,route_number,source,destination,endpoint,nearest_stop_name,distance_m,component_count,geometry_type
0,118N,Tambaram West Bus Stand,Neelamangalam,end,NEELAMANGALAM,1.70,1,MultiLineString
1,118N,Tambaram West Bus Stand,Neelamangalam,start,TAMBARAM WEST BUS STAND,333.41,1,MultiLineString
2,120E,Kilkondaiyur,Broadway,end,BROADWAY,24.51,1,MultiLineString
3,120E,Kilkondaiyur,Broadway,start,KAVANUR,1609.98,1,MultiLineString
4,120KET,Kilkondaiyur,Broadway,end,BROADWAY,18.61,1,MultiLineString
5,120KET,Kilkondaiyur,Broadway,start,KAVANUR,1609.98,1,MultiLineString
6,555M,Tambaram West Bus Stand,Thiruporur Bus Stand,end,THIRUPORUR BUS STAND,3.13,1,MultiLineString
7,555M,Tambaram West Bus Stand,Thiruporur Bus Stand,start,TAMBARAM WEST BUS STAND,104.95,1,MultiLineString
8,S12,Velachery Railway Station,Guindy Railway Station,end,GUINDY RAILWAY STATION,2.48,1,MultiLineString
9,S12,Velachery Railway Station,Guindy Railway Station,start,VELACHERY RAILWAY STATION,144.03,1,MultiLineString


In [42]:
routes_to_compare = ["120KET", "120E"]

route_geometry_check = []

for route_number in routes_to_compare:

    rows = geo_route_name_df[
        geo_route_name_df["properties.route_name"]
        .astype(str).str.strip().str.upper()
        == route_number
    ]

    if rows.empty:
        print("Route not found:", route_number)
        continue

    row = rows.iloc[0]
    components = row["geometry.coordinates"]

    route_geometry_check.append({
        "route_number": route_number,
        "source": row["properties.source"],
        "destination": row["properties.destinatio"],
        "geometry_type": row["geometry.type"],
        "component_count": len(components),
        "coordinate_count": sum(len(line) for line in components),
        "first_coordinate": components[0][0],
        "last_coordinate": components[-1][-1]
    })

display(pd.DataFrame(route_geometry_check))

,route_number,source,destination,geometry_type,component_count,coordinate_count,first_coordinate,last_coordinate
0,120KET,Kilkondaiyur,Broadway,MultiLineString,1,811,"[80.04425, 13.18]","[80.28372, 13.08818]"
1,120E,Kilkondaiyur,Broadway,MultiLineString,1,783,"[80.04425, 13.18]","[80.28372, 13.08806]"


In [43]:
import pandas as pd
import numpy as np

routes_to_check = [
    "118N", "120E", "120KET",
    "555M", "S12", "S54TCT"
]

# GTFS tables
routes_df = gtfs_data["routes"].copy()
trips_df = gtfs_data["trips"].copy()
stop_times_df = gtfs_data["stop_times"].copy()
stops_df = gtfs_data["stops"].copy()

# Make route IDs and stop IDs consistent
for df, col in [
    (routes_df, "route_id"),
    (trips_df, "route_id"),
    (trips_df, "trip_id"),
    (stop_times_df, "trip_id"),
    (stop_times_df, "stop_id"),
    (stops_df, "stop_id")
]:
    df[col] = df[col].astype(str).str.strip()

# Numeric stop sequence for ordering
stop_times_df["stop_sequence_num"] = pd.to_numeric(
    stop_times_df["stop_sequence"],
    errors="coerce"
)

# Find GTFS route IDs for the six route numbers
selected_routes = routes_df[
    routes_df["route_short_name"]
    .astype(str).str.strip().str.upper()
    .isin(routes_to_check)
].copy()

# Get trips belonging to those route IDs
selected_trips = trips_df[
    trips_df["route_id"].isin(selected_routes["route_id"])
].copy()

# Attach stop names and coordinates
trip_stops = selected_trips[
    ["route_id", "trip_id"]
].merge(
    stop_times_df[
        ["trip_id", "stop_id", "stop_sequence_num"]
    ],
    on="trip_id",
    how="inner"
).merge(
    stops_df[
        ["stop_id", "stop_name", "stop_lat", "stop_lon"]
    ],
    on="stop_id",
    how="left"
)

# Sort stops within each trip
trip_stops = trip_stops.sort_values(
    ["trip_id", "stop_sequence_num"]
)

# Extract first and last stop of every trip
first_stops = trip_stops.groupby(
    "trip_id", as_index=False
).first()

last_stops = trip_stops.groupby(
    "trip_id", as_index=False
).last()

first_stops["terminal_type"] = "first_stop"
last_stops["terminal_type"] = "last_stop"

trip_terminals = pd.concat(
    [first_stops, last_stops],
    ignore_index=True
)

# Attach route number and route long name
trip_terminals = trip_terminals.merge(
    selected_routes[
        ["route_id", "route_short_name", "route_long_name"]
    ],
    on="route_id",
    how="left"
)

# Display terminal stop patterns
terminal_summary = (
    trip_terminals.groupby(
        [
            "route_short_name",
            "route_id",
            "terminal_type",
            "stop_name"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="trip_count")
    .sort_values(
        ["route_short_name", "terminal_type", "trip_count"],
        ascending=[True, True, False]
    )
)

display(terminal_summary)

,route_short_name,route_id,terminal_type,stop_name,trip_count
0,118N,2146,first_stop,TAMBARAM WEST BUS STAND,8
2,118N,2147,first_stop,NEELAMANGALAM,8
1,118N,2146,last_stop,NEELAMANGALAM,8
3,118N,2147,last_stop,TAMBARAM WEST BUS STAND,8
8,120E,2189,first_stop,AVADI B.T,4
10,120E,2190,first_stop,KILKONDAIYUR,4
4,120E,2187,first_stop,AMBATTUR I.E,1
6,120E,2188,first_stop,KILKONDAIYUR,1
12,120E,2191,first_stop,BROADWAY,1
14,120E,2192,first_stop,KILKONDAIYUR,1


In [44]:
import pandas as pd
import numpy as np

def haversine_m(lat1, lon1, lat2, lon2):
    earth_radius = 6371000

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    return 2 * earth_radius * np.arcsin(
        np.sqrt(np.clip(a, 0, 1))
    )


routes_to_check = [
    "118N", "120E", "120KET",
    "555M", "S12", "S54TCT"
]

# Summarise distinct GTFS terminal stops and their trip counts
terminal_candidates = (
    trip_terminals.groupby(
        [
            "route_short_name",
            "route_id",
            "terminal_type",
            "stop_id",
            "stop_name",
            "stop_lat",
            "stop_lon"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="trip_count")
)

terminal_candidates["stop_lat"] = pd.to_numeric(
    terminal_candidates["stop_lat"], errors="coerce"
)
terminal_candidates["stop_lon"] = pd.to_numeric(
    terminal_candidates["stop_lon"], errors="coerce"
)

results = []

for route_number in routes_to_check:

    geo_rows = geo_route_name_df[
        geo_route_name_df["properties.route_name"]
        .astype(str).str.strip().str.upper()
        == route_number.upper()
    ]

    if geo_rows.empty:
        continue

    geo_row = geo_rows.iloc[0]
    coordinates = geo_row["geometry.coordinates"]

    if geo_row["geometry.type"] == "MultiLineString":
        components = coordinates
    else:
        components = [coordinates]

    endpoints = {
        "start": components[0][0],
        "end": components[-1][-1]
    }

    candidates = terminal_candidates[
        terminal_candidates["route_short_name"]
        .astype(str).str.strip().str.upper()
        == route_number.upper()
    ].dropna(subset=["stop_lat", "stop_lon"])

    for endpoint_type, point in endpoints.items():

        lon, lat = point

        if candidates.empty:
            continue

        distances = haversine_m(
            lat,
            lon,
            candidates["stop_lat"].to_numpy(dtype=float),
            candidates["stop_lon"].to_numpy(dtype=float)
        )

        nearest_indices = np.argsort(distances)[:3]

        for rank, idx in enumerate(nearest_indices, start=1):

            candidate = candidates.iloc[idx]

            results.append({
                "route_number": route_number,
                "geometry_endpoint": endpoint_type,
                "candidate_rank": rank,
                "GTFS_route_id": candidate["route_id"],
                "terminal_type": candidate["terminal_type"],
                "GTFS_terminal_name": candidate["stop_name"],
                "GTFS_stop_id": candidate["stop_id"],
                "trip_count": candidate["trip_count"],
                "distance_m": round(float(distances[idx]), 2)
            })


terminal_coordinate_check = pd.DataFrame(results)

display(
    terminal_coordinate_check.sort_values(
        ["route_number", "geometry_endpoint", "candidate_rank"]
    ).reset_index(drop=True)
)

,route_number,geometry_endpoint,candidate_rank,GTFS_route_id,terminal_type,GTFS_terminal_name,GTFS_stop_id,trip_count,distance_m
0,118N,end,1,2146,last_stop,NEELAMANGALAM,WronNRym,8,2.17
1,118N,end,2,2147,first_stop,NEELAMANGALAM,hxZYrmAJ,8,4.87
2,118N,end,3,2146,first_stop,TAMBARAM WEST BUS STAND,ojqTFFWy,8,11957.32
3,118N,start,1,2146,first_stop,TAMBARAM WEST BUS STAND,ojqTFFWy,8,365.91
4,118N,start,2,2147,last_stop,TAMBARAM WEST BUS STAND,ojqTFFWy,8,365.91
5,118N,start,3,2147,first_stop,NEELAMANGALAM,hxZYrmAJ,8,12316.15
6,120E,end,1,2187,first_stop,AMBATTUR I.E,8,1,61755.93
7,120E,end,2,2187,last_stop,KILKONDAIYUR,801,1,61755.93
8,120E,end,3,2188,first_stop,KILKONDAIYUR,801,1,61755.93
9,120E,start,1,2187,first_stop,AMBATTUR I.E,8,1,70918.24


In [45]:
routes_to_check = [
    "118N", "120E", "120KET",
    "555M", "S12", "S54TCT"
]

# Get exact route-number and origin-destination matches
exact_route_records = all_route_matches[
    all_route_matches["route_name"]
    .astype(str).str.strip().str.upper()
    .isin(routes_to_check)
].copy()

# Display the GTFS records that actually matched each GeoJSON route
exact_route_records = exact_route_records[
    [
        "route_name",
        "source",
        "destination",
        "route_id",
        "route_short_name",
        "route_long_name",
        "gtfs_origin",
        "gtfs_destination",
        "match_direction"
    ]
].sort_values(
    ["route_name", "route_id"]
)

display(exact_route_records.reset_index(drop=True))

print(
    "Distinct GTFS route IDs:",
    exact_route_records["route_id"].nunique()
)

print(
    "Distinct GeoJSON route features:",
    exact_route_records["route_name"].nunique()
)

,route_name,source,destination,route_id,route_short_name,route_long_name,gtfs_origin,gtfs_destination,match_direction
0,118N,Tambaram West Bus Stand,Neelamangalam,2146,118N,TAMBARAM WEST BUS STAND To NEELAMANGALAM,TAMBARAM WEST BUS STAND,NEELAMANGALAM,same_direction
1,118N,Tambaram West Bus Stand,Neelamangalam,2147,118N,NEELAMANGALAM To TAMBARAM WEST BUS STAND,NEELAMANGALAM,TAMBARAM WEST BUS STAND,reverse_direction
2,120E,Kilkondaiyur,Broadway,2191,120E,BROADWAY To KILKONDAIYUR,BROADWAY,KILKONDAIYUR,reverse_direction
3,120E,Kilkondaiyur,Broadway,2192,120E,KILKONDAIYUR To BROADWAY,KILKONDAIYUR,BROADWAY,same_direction
4,120KET,Kilkondaiyur,Broadway,2213,120KET,BROADWAY To KILKONDAIYUR,BROADWAY,KILKONDAIYUR,reverse_direction
5,120KET,Kilkondaiyur,Broadway,2214,120KET,KILKONDAIYUR To BROADWAY,KILKONDAIYUR,BROADWAY,same_direction
6,555M,Tambaram West Bus Stand,Thiruporur Bus Stand,3328,555M,TAMBARAM WEST BUS STAND To THIRUPORUR BUS STAND,TAMBARAM WEST BUS STAND,THIRUPORUR BUS STAND,same_direction
7,555M,Tambaram West Bus Stand,Thiruporur Bus Stand,3329,555M,THIRUPORUR BUS STAND To TAMBARAM WEST BUS STAND,THIRUPORUR BUS STAND,TAMBARAM WEST BUS STAND,reverse_direction
8,S12,Velachery Railway Station,Guindy Railway Station,692,S12,GUINDY RAILWAY STATION To VELACHERY RAILWAY ST...,GUINDY RAILWAY STATION,VELACHERY RAILWAY STATION,reverse_direction
9,S12,Velachery Railway Station,Guindy Railway Station,695,S12,VELACHERY RAILWAY STATION To GUINDY RAILWAY ST...,VELACHERY RAILWAY STATION,GUINDY RAILWAY STATION,same_direction


Distinct GTFS route IDs: 12
Distinct GeoJSON route features: 6


In [46]:
matched_route_ids = (
    exact_route_records["route_id"]
    .astype(str)
    .unique()
)

matched_gtfs_routes = routes_df[
    routes_df["route_id"].astype(str).isin(matched_route_ids)
].copy()

matched_trip_ids = trips_df[
    trips_df["route_id"].astype(str).isin(matched_route_ids)
]

matched_terminal_stops = trip_stops[
    trip_stops["route_id"].astype(str).isin(matched_route_ids)
].copy()

matched_terminal_summary = (
    trip_terminals[
        trip_terminals["route_id"].astype(str).isin(matched_route_ids)
    ]
    .groupby(
        [
            "route_short_name",
            "route_id",
            "terminal_type",
            "stop_name"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="trip_count")
    .sort_values(
        ["route_short_name", "route_id", "terminal_type"]
    )
)

display(matched_gtfs_routes[
    ["route_id", "route_short_name", "route_long_name"]
].sort_values(["route_short_name", "route_id"]))

display(matched_terminal_summary)

,route_id,route_short_name,route_long_name
1014,2146,118N,TAMBARAM WEST BUS STAND To NEELAMANGALAM
1015,2147,118N,NEELAMANGALAM To TAMBARAM WEST BUS STAND
1057,2191,120E,BROADWAY To KILKONDAIYUR
1058,2192,120E,KILKONDAIYUR To BROADWAY
1079,2213,120KET,BROADWAY To KILKONDAIYUR
1080,2214,120KET,KILKONDAIYUR To BROADWAY
2049,3328,555M,TAMBARAM WEST BUS STAND To THIRUPORUR BUS STAND
2050,3329,555M,THIRUPORUR BUS STAND To TAMBARAM WEST BUS STAND
391,692,S12,GUINDY RAILWAY STATION To VELACHERY RAILWAY ST...
392,695,S12,VELACHERY RAILWAY STATION To GUINDY RAILWAY ST...


,route_short_name,route_id,terminal_type,stop_name,trip_count
0,118N,2146,first_stop,TAMBARAM WEST BUS STAND,8
1,118N,2146,last_stop,NEELAMANGALAM,8
2,118N,2147,first_stop,NEELAMANGALAM,8
3,118N,2147,last_stop,TAMBARAM WEST BUS STAND,8
4,120E,2191,first_stop,BROADWAY,1
5,120E,2191,last_stop,KILKONDAIYUR,1
6,120E,2192,first_stop,KILKONDAIYUR,1
7,120E,2192,last_stop,BROADWAY,1
8,120KET,2213,first_stop,BROADWAY,1
9,120KET,2213,last_stop,KILKONDAIYUR,1


In [47]:
import numpy as np
import pandas as pd

# Prepare exact GTFS terminal records
matched_route_ids = (
    exact_route_records["route_id"].astype(str).unique()
)

matched_trip_ids = trips_df[
    trips_df["route_id"].astype(str).isin(matched_route_ids)
]

matched_stop_times = stop_times_df[
    stop_times_df["trip_id"].isin(matched_trip_ids["trip_id"])
].copy()

matched_stop_times["stop_sequence_num"] = pd.to_numeric(
    matched_stop_times["stop_sequence"],
    errors="coerce"
)

matched_trip_stops = matched_stop_times.merge(
    stops_df[
        ["stop_id", "stop_name", "stop_lat", "stop_lon"]
    ],
    on="stop_id",
    how="left"
)

matched_trip_stops = matched_trip_stops.sort_values(
    ["trip_id", "stop_sequence_num"]
)

first_terminal = matched_trip_stops.groupby(
    "trip_id", as_index=False
).first()

last_terminal = matched_trip_stops.groupby(
    "trip_id", as_index=False
).last()

first_terminal["terminal_type"] = "first_stop"
last_terminal["terminal_type"] = "last_stop"

exact_terminals = pd.concat(
    [first_terminal, last_terminal],
    ignore_index=True
)

exact_terminals = exact_terminals.merge(
    trips_df[["trip_id", "route_id"]],
    on="trip_id",
    how="left"
)

exact_terminals = exact_terminals.merge(
    routes_df[
        ["route_id", "route_short_name"]
    ],
    on="route_id",
    how="left"
)

# Haversine function
def haversine_m(lat1, lon1, lat2, lon2):
    radius = 6371000

    lat1, lon1 = np.radians(lat1), np.radians(lon1)
    lat2, lon2 = np.radians(lat2), np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    return 2 * radius * np.arcsin(
        np.sqrt(np.clip(a, 0, 1))
    )


geometry_terminal_results = []

for _, route in exact_route_records.iterrows():

    route_number = str(route["route_name"]).strip()
    route_id = str(route["route_id"])

    geo_row = geo_route_name_df[
        geo_route_name_df["properties.route_name"]
        .astype(str).str.strip().str.upper()
        == route_number.upper()
    ].iloc[0]

    components = geo_row["geometry.coordinates"]

    if geo_row["geometry.type"] == "MultiLineString":
        geometry_start = components[0][0]
        geometry_end = components[-1][-1]
    else:
        geometry_start = components[0]
        geometry_end = components[-1]

    # Match only the exact GTFS route ID
    route_terminals = exact_terminals[
        exact_terminals["route_id"].astype(str) == route_id
    ]

    # Use the most frequent first and last terminal names
    for terminal_type, geometry_point in [
        ("first_stop", geometry_start),
        ("last_stop", geometry_end)
    ]:

        candidates = route_terminals[
            route_terminals["terminal_type"] == terminal_type
        ].copy()

        if candidates.empty:
            continue

        candidates["stop_lat"] = pd.to_numeric(
            candidates["stop_lat"], errors="coerce"
        )
        candidates["stop_lon"] = pd.to_numeric(
            candidates["stop_lon"], errors="coerce"
        )

        candidates = candidates.dropna(
            subset=["stop_lat", "stop_lon"]
        )

        if candidates.empty:
            continue

        # Aggregate repeated trips by terminal stop
        terminal_counts = (
            candidates.groupby(
                ["stop_id", "stop_name", "stop_lat", "stop_lon"]
            )
            .size()
            .reset_index(name="trip_count")
        )

        lon, lat = geometry_point

        terminal_counts["distance_m"] = haversine_m(
            lat,
            lon,
            terminal_counts["stop_lat"].to_numpy(),
            terminal_counts["stop_lon"].to_numpy()
        )

        nearest = terminal_counts.sort_values(
            "distance_m"
        ).iloc[0]

        geometry_terminal_results.append({
            "route_number": route_number,
            "route_id": route_id,
            "match_direction": route["match_direction"],
            "terminal_type": terminal_type,
            "GTFS_terminal": nearest["stop_name"],
            "trip_count": int(nearest["trip_count"]),
            "distance_m": round(
                float(nearest["distance_m"]), 2
            )
        })


geometry_terminal_check = pd.DataFrame(
    geometry_terminal_results
)

display(
    geometry_terminal_check.sort_values(
        ["route_number", "route_id", "terminal_type"]
    ).reset_index(drop=True)
)

,route_number,route_id,match_direction,terminal_type,GTFS_terminal,trip_count,distance_m
0,118N,2146,same_direction,first_stop,TAMBARAM WEST BUS STAND,8,365.91
1,118N,2146,same_direction,last_stop,NEELAMANGALAM,8,2.17
2,118N,2147,reverse_direction,first_stop,NEELAMANGALAM,8,12316.15
3,118N,2147,reverse_direction,last_stop,TAMBARAM WEST BUS STAND,8,11957.32
4,120E,2191,reverse_direction,first_stop,BROADWAY,1,70918.24
5,120E,2191,reverse_direction,last_stop,KILKONDAIYUR,1,61755.93
6,120E,2192,same_direction,first_stop,KILKONDAIYUR,1,70918.24
7,120E,2192,same_direction,last_stop,BROADWAY,1,61755.93
8,120KET,2213,reverse_direction,first_stop,BROADWAY,1,70918.24
9,120KET,2213,reverse_direction,last_stop,KILKONDAIYUR,1,61768.86


In [48]:
corrected_results = []

# Only compare geometry with its matching direction
forward_records = exact_route_records[
    exact_route_records["match_direction"] == "same_direction"
].copy()

for _, route in forward_records.iterrows():

    route_number = str(route["route_name"]).strip()
    route_id = str(route["route_id"])

    geo_row = geo_route_name_df[
        geo_route_name_df["properties.route_name"]
        .astype(str).str.strip().str.upper()
        == route_number.upper()
    ].iloc[0]

    coordinates = geo_row["geometry.coordinates"]

    if geo_row["geometry.type"] == "MultiLineString":
        geometry_start = coordinates[0][0]
        geometry_end = coordinates[0][-1]
    else:
        geometry_start = coordinates[0]
        geometry_end = coordinates[-1]

    route_terminals = exact_terminals[
        exact_terminals["route_id"].astype(str) == route_id
    ]

    for terminal_type, geometry_point in [
        ("first_stop", geometry_start),
        ("last_stop", geometry_end)
    ]:

        candidates = route_terminals[
            route_terminals["terminal_type"] == terminal_type
        ].copy()

        candidates["stop_lat"] = pd.to_numeric(
            candidates["stop_lat"], errors="coerce"
        )
        candidates["stop_lon"] = pd.to_numeric(
            candidates["stop_lon"], errors="coerce"
        )

        candidates = candidates.dropna(
            subset=["stop_lat", "stop_lon"]
        )

        terminal_counts = (
            candidates.groupby(
                ["stop_id", "stop_name", "stop_lat", "stop_lon"]
            )
            .size()
            .reset_index(name="trip_count")
        )

        # GeoJSON coordinate order is [longitude, latitude]
        geometry_lon, geometry_lat = geometry_point

        terminal_counts["distance_m"] = haversine_m(
            geometry_lat,
            geometry_lon,
            terminal_counts["stop_lat"].to_numpy(),
            terminal_counts["stop_lon"].to_numpy()
        )

        nearest = terminal_counts.sort_values(
            "distance_m"
        ).iloc[0]

        corrected_results.append({
            "route_number": route_number,
            "route_id": route_id,
            "terminal_type": terminal_type,
            "GTFS_terminal": nearest["stop_name"],
            "trip_count": int(nearest["trip_count"]),
            "distance_m": round(
                float(nearest["distance_m"]), 2
            )
        })

corrected_geometry_check = pd.DataFrame(corrected_results)

display(
    corrected_geometry_check.sort_values(
        ["route_number", "terminal_type"]
    ).reset_index(drop=True)
)

,route_number,route_id,terminal_type,GTFS_terminal,trip_count,distance_m
0,118N,2146,first_stop,TAMBARAM WEST BUS STAND,8,365.91
1,118N,2146,last_stop,NEELAMANGALAM,8,2.17
2,120E,2192,first_stop,KILKONDAIYUR,1,70918.24
3,120E,2192,last_stop,BROADWAY,1,61755.93
4,120KET,2214,first_stop,KILKONDAIYUR,1,70918.24
5,120KET,2214,last_stop,BROADWAY,1,61768.86
6,555M,3328,first_stop,TAMBARAM WEST BUS STAND,5,136.76
7,555M,3328,last_stop,THIRUPORUR BUS STAND,5,0.00
8,S12,695,first_stop,VELACHERY RAILWAY STATION,33,335.32
9,S12,695,last_stop,GUINDY RAILWAY STATION,33,3.94


In [49]:
coordinate_diagnostics = []

for _, route in forward_records.iterrows():

    route_number = str(route["route_name"]).strip()
    route_id = str(route["route_id"])

    geo_row = geo_route_name_df[
        geo_route_name_df["properties.route_name"]
        .astype(str).str.strip().str.upper()
        == route_number.upper()
    ].iloc[0]

    coordinates = geo_row["geometry.coordinates"]

    if geo_row["geometry.type"] == "MultiLineString":
        geometry_start = coordinates[0][0]
        geometry_end = coordinates[0][-1]
    else:
        geometry_start = coordinates[0]
        geometry_end = coordinates[-1]

    for terminal_type, geometry_point in [
        ("first_stop", geometry_start),
        ("last_stop", geometry_end)
    ]:

        terminal = exact_terminals[
            (exact_terminals["route_id"].astype(str) == route_id) &
            (exact_terminals["terminal_type"] == terminal_type)
        ].copy()

        terminal = terminal.drop_duplicates(
            subset=["stop_id"]
        )

        terminal = terminal.merge(
            prepared_stops[
                ["stop_id", "map_lat", "map_lon", "coordinate_status"]
            ],
            on="stop_id",
            how="left"
        )

        terminal["stop_lat"] = pd.to_numeric(
            terminal["stop_lat"], errors="coerce"
        )
        terminal["stop_lon"] = pd.to_numeric(
            terminal["stop_lon"], errors="coerce"
        )

        geometry_lon, geometry_lat = geometry_point

        for _, stop in terminal.iterrows():

            gtfs_distance = haversine_m(
                geometry_lat,
                geometry_lon,
                stop["stop_lat"],
                stop["stop_lon"]
            )

            if pd.notna(stop["map_lat"]) and pd.notna(stop["map_lon"]):
                map_distance = haversine_m(
                    geometry_lat,
                    geometry_lon,
                    stop["map_lat"],
                    stop["map_lon"]
                )
            else:
                map_distance = np.nan

            coordinate_diagnostics.append({
                "route": route_number,
                "route_id": route_id,
                "terminal_type": terminal_type,
                "stop_id": stop["stop_id"],
                "stop_name": stop["stop_name"],
                "GTFS_lat": stop["stop_lat"],
                "GTFS_lon": stop["stop_lon"],
                "map_lat": stop["map_lat"],
                "map_lon": stop["map_lon"],
                "coordinate_status": stop["coordinate_status"],
                "GTFS_distance_m": round(float(gtfs_distance), 2),
                "prepared_map_distance_m": (
                    round(float(map_distance), 2)
                    if pd.notna(map_distance) else np.nan
                )
            })

coordinate_diagnostics_df = pd.DataFrame(
    coordinate_diagnostics
)

display(
    coordinate_diagnostics_df.sort_values(
        ["route", "terminal_type", "GTFS_distance_m"]
    ).reset_index(drop=True)
)

,route,route_id,terminal_type,stop_id,stop_name,GTFS_lat,GTFS_lon,map_lat,map_lon,coordinate_status,GTFS_distance_m,prepared_map_distance_m
0,118N,2146,first_stop,ojqTFFWy,TAMBARAM WEST BUS STAND,12.927610,80.118670,12.927652,80.118626,verified_geojson_match,365.91,365.80
1,118N,2146,last_stop,WronNRym,NEELAMANGALAM,12.858110,80.034470,12.858104,80.034476,verified_geojson_match,2.17,1.70
2,120E,2192,first_stop,801,KILKONDAIYUR,12.549663,80.143925,NaN,NaN,placeholder_candidate,70918.24,NaN
3,120E,2192,last_stop,22,BROADWAY,12.549663,80.143925,NaN,NaN,placeholder_candidate,61755.93,NaN
4,120KET,2214,first_stop,801,KILKONDAIYUR,12.549663,80.143925,NaN,NaN,placeholder_candidate,70918.24,NaN
5,120KET,2214,last_stop,22,BROADWAY,12.549663,80.143925,NaN,NaN,placeholder_candidate,61768.86,NaN
6,555M,3328,first_stop,ojqTFFWy,TAMBARAM WEST BUS STAND,12.927610,80.118670,12.927652,80.118626,verified_geojson_match,136.76,136.71
7,555M,3328,last_stop,wcUGKfLa,THIRUPORUR BUS STAND,12.722200,80.187300,12.722195,80.187328,verified_geojson_match,0.00,3.13
8,S12,695,first_stop,okaePkJr,VELACHERY RAILWAY STATION,12.971200,80.218990,12.971216,80.218923,verified_geojson_match,335.32,334.89
9,S12,695,last_stop,VfTFpfzB,GUINDY RAILWAY STATION,13.007900,80.213060,13.007933,80.213012,verified_geojson_match,3.94,2.48


In [50]:
nearest_endpoint_results = []

map_stops = prepared_stops[
    prepared_stops["include_in_default_map"] == True
].copy()

map_stops["map_lat"] = pd.to_numeric(
    map_stops["map_lat"], errors="coerce"
)
map_stops["map_lon"] = pd.to_numeric(
    map_stops["map_lon"], errors="coerce"
)

map_stops = map_stops.dropna(
    subset=["map_lat", "map_lon"]
)

for _, route in forward_records.iterrows():

    route_number = str(route["route_name"]).strip()

    geo_row = geo_route_name_df[
        geo_route_name_df["properties.route_name"]
        .astype(str).str.strip().str.upper()
        == route_number.upper()
    ].iloc[0]

    coordinates = geo_row["geometry.coordinates"]

    if geo_row["geometry.type"] == "MultiLineString":
        geometry_start = coordinates[0][0]
        geometry_end = coordinates[0][-1]
    else:
        geometry_start = coordinates[0]
        geometry_end = coordinates[-1]

    for endpoint_name, point in [
        ("geometry_start", geometry_start),
        ("geometry_end", geometry_end)
    ]:

        geometry_lon, geometry_lat = point

        distances = haversine_m(
            geometry_lat,
            geometry_lon,
            map_stops["map_lat"].to_numpy(),
            map_stops["map_lon"].to_numpy()
        )

        nearest = map_stops.copy()
        nearest["distance_m"] = distances

        nearest = nearest.nsmallest(3, "distance_m")

        for rank, (_, stop) in enumerate(
            nearest.iterrows(), start=1
        ):
            nearest_endpoint_results.append({
                "route": route_number,
                "endpoint": endpoint_name,
                "nearest_rank": rank,
                "stop_name": stop["stop_name"],
                "stop_id": stop["stop_id"],
                "coordinate_status": stop["coordinate_status"],
                "distance_m": round(
                    float(stop["distance_m"]), 2
                )
            })

nearest_endpoint_df = pd.DataFrame(
    nearest_endpoint_results
)

display(
    nearest_endpoint_df.sort_values(
        ["route", "endpoint", "nearest_rank"]
    ).reset_index(drop=True)
)

,route,endpoint,nearest_rank,stop_name,stop_id,coordinate_status,distance_m
0,118N,geometry_end,1,NEELAMANGALAM,WronNRym,verified_geojson_match,1.70
1,118N,geometry_end,2,NEELAMANGALAM,hxZYrmAJ,verified_geojson_match,3.04
2,118N,geometry_end,3,BHARATHIYAR MATRICULATION SCHOOL,AidKYcdB,verified_geojson_match,428.89
3,118N,geometry_start,1,TAMBARAM WEST BUS STAND,60b1deb61b7391af4c7c2a2017479dd2,unique_unmatched_provisional,333.41
4,118N,geometry_start,2,TAMBARAM WEST BUS STAND,a4f1c432,unique_unmatched_provisional,353.56
5,118N,geometry_start,3,TAMBARAM WEST BUS STAND,ejMjPdTH,verified_geojson_match,353.63
6,120E,geometry_end,1,BROADWAY,EkWBPXmr,verified_geojson_match,24.51
7,120E,geometry_end,2,BROADWAY,134ee62a,unique_unmatched_provisional,34.66
8,120E,geometry_end,3,BROADWAY,390015f1e395909e465ef0281d8df3c4,unique_unmatched_provisional,45.08
9,120E,geometry_start,1,KAVANUR,TbetMXoD,verified_geojson_match,1609.98


In [51]:
# Expected terminal names for each forward route
expected_terminals = {
    "118N": ["TAMBARAM WEST", "NEELAMANGALAM"],
    "120E": ["KILKONDAIYUR", "BROADWAY"],
    "120KET": ["KILKONDAIYUR", "BROADWAY"],
    "555M": ["TAMBARAM WEST", "THIRUPORUR"],
    "S12": ["VELACHERY RAILWAY STATION", "GUINDY RAILWAY STATION"],
    "S54TCT": ["PORUR", "CHEMBARAMBAKKAM"]
}

terminal_name_results = []

for _, route in forward_records.iterrows():

    route_number = str(route["route_name"]).strip()

    geo_row = geo_route_name_df[
        geo_route_name_df["properties.route_name"]
        .astype(str).str.strip().str.upper()
        == route_number.upper()
    ].iloc[0]

    coordinates = geo_row["geometry.coordinates"]

    if geo_row["geometry.type"] == "MultiLineString":
        start_point = coordinates[0][0]
        end_point = coordinates[0][-1]
    else:
        start_point = coordinates[0]
        end_point = coordinates[-1]

    endpoints = [
        ("geometry_start", start_point),
        ("geometry_end", end_point)
    ]

    for (endpoint_name, point), expected_name in zip(
        endpoints, expected_terminals[route_number]
    ):

        candidates = prepared_stops[
            prepared_stops["stop_name"]
            .astype(str).str.upper()
            .str.contains(expected_name, regex=False, na=False)
        ].copy()

        candidates["map_lat"] = pd.to_numeric(
            candidates["map_lat"], errors="coerce"
        )
        candidates["map_lon"] = pd.to_numeric(
            candidates["map_lon"], errors="coerce"
        )

        candidates = candidates.dropna(
            subset=["map_lat", "map_lon"]
        )

        if candidates.empty:
            terminal_name_results.append({
                "route": route_number,
                "endpoint": endpoint_name,
                "expected_name": expected_name,
                "matching_stop": "NO MAP-READY MATCH",
                "coordinate_status": None,
                "distance_m": np.nan
            })
            continue

        geometry_lon, geometry_lat = point

        candidates["distance_m"] = haversine_m(
            geometry_lat,
            geometry_lon,
            candidates["map_lat"].to_numpy(),
            candidates["map_lon"].to_numpy()
        )

        for _, candidate in candidates.nsmallest(
            3, "distance_m"
        ).iterrows():

            terminal_name_results.append({
                "route": route_number,
                "endpoint": endpoint_name,
                "expected_name": expected_name,
                "matching_stop": candidate["stop_name"],
                "stop_id": candidate["stop_id"],
                "coordinate_status": candidate["coordinate_status"],
                "distance_m": round(
                    float(candidate["distance_m"]), 2
                )
            })

terminal_name_check = pd.DataFrame(terminal_name_results)

display(
    terminal_name_check.sort_values(
        ["route", "endpoint", "distance_m"]
    ).reset_index(drop=True)
)

,route,endpoint,expected_name,matching_stop,stop_id,coordinate_status,distance_m
0,118N,geometry_end,NEELAMANGALAM,NEELAMANGALAM,WronNRym,verified_geojson_match,1.70
1,118N,geometry_end,NEELAMANGALAM,NEELAMANGALAM,hxZYrmAJ,verified_geojson_match,3.04
2,118N,geometry_start,TAMBARAM WEST,TAMBARAM WEST BUS STAND,60b1deb61b7391af4c7c2a2017479dd2,unique_unmatched_provisional,333.41
3,118N,geometry_start,TAMBARAM WEST,TAMBARAM WEST BUS STAND,a4f1c432,unique_unmatched_provisional,353.56
4,118N,geometry_start,TAMBARAM WEST,TAMBARAM WEST BUS STAND,ejMjPdTH,verified_geojson_match,353.63
5,120E,geometry_end,BROADWAY,BROADWAY,EkWBPXmr,verified_geojson_match,24.51
6,120E,geometry_end,BROADWAY,BROADWAY,134ee62a,unique_unmatched_provisional,34.66
7,120E,geometry_end,BROADWAY,BROADWAY,390015f1e395909e465ef0281d8df3c4,unique_unmatched_provisional,45.08
8,120E,geometry_start,KILKONDAIYUR,NO MAP-READY MATCH,NaN,None,NaN
9,120KET,geometry_end,BROADWAY,BROADWAY,EkWBPXmr,verified_geojson_match,18.61


In [52]:
kilkondaiyur_stops = prepared_stops[
    prepared_stops["stop_name"]
    .astype(str)
    .str.upper()
    .str.contains("KILKONDAIYUR", na=False)
].copy()

display(
    kilkondaiyur_stops[
        [
            "stop_id",
            "stop_name",
            "stop_lat",
            "stop_lon",
            "map_lat",
            "map_lon",
            "coordinate_status",
            "include_in_default_map"
        ]
    ].reset_index(drop=True)
)

print("Matching stops:", len(kilkondaiyur_stops))

,stop_id,stop_name,stop_lat,stop_lon,map_lat,map_lon,coordinate_status,include_in_default_map
0,801,KILKONDAIYUR,12.549663,80.143925,NaN,NaN,placeholder_candidate,False


Matching stops: 1


In [53]:
route_selection = pd.DataFrame([
    {
        "route_name": "118N",
        "route_id": "2146",
        "source": "Tambaram West Bus Stand",
        "destination": "Neelamangalam",
        "geometry_status": "included_with_warning",
        "quality_note": "Start endpoint approximately 333 m from nearest map-ready Tambaram West stop."
    },
    {
        "route_name": "120E",
        "route_id": "2192",
        "source": "Kilkondaiyur",
        "destination": "Broadway",
        "geometry_status": "included_with_warning",
        "quality_note": "Kilkondaiyur GTFS coordinates are placeholders; no map-ready terminal coordinate."
    },
    {
        "route_name": "120KET",
        "route_id": "2214",
        "source": "Kilkondaiyur",
        "destination": "Broadway",
        "geometry_status": "included_with_warning",
        "quality_note": "Kilkondaiyur GTFS coordinates are placeholders; no map-ready terminal coordinate."
    },
    {
        "route_name": "555M",
        "route_id": "3328",
        "source": "Tambaram West Bus Stand",
        "destination": "Thiruporur Bus Stand",
        "geometry_status": "included",
        "quality_note": "Both endpoints have nearby matching map-ready stops."
    },
    {
        "route_name": "S12",
        "route_id": "695",
        "source": "Velachery Railway Station",
        "destination": "Guindy Railway Station",
        "geometry_status": "included_with_warning",
        "quality_note": "Start endpoint approximately 144 m from nearest matching map-ready stop."
    },
    {
        "route_name": "S54TCT",
        "route_id": "4580",
        "source": "Porur",
        "destination": "Chembarambakkam",
        "geometry_status": "included_with_warning",
        "quality_note": "Destination endpoint approximately 989 m from nearest matching map-ready Chembarambakkam stop."
    }
])

display(route_selection)

print("Selected routes:", len(route_selection))
print(
    "Included without warning:",
    (route_selection["geometry_status"] == "included").sum()
)
print(
    "Included with warning:",
    (route_selection["geometry_status"] == "included_with_warning").sum()
)

,route_name,route_id,source,destination,geometry_status,quality_note
0,118N,2146,Tambaram West Bus Stand,Neelamangalam,included_with_warning,Start endpoint approximately 333 m from neares...
1,120E,2192,Kilkondaiyur,Broadway,included_with_warning,Kilkondaiyur GTFS coordinates are placeholders...
2,120KET,2214,Kilkondaiyur,Broadway,included_with_warning,Kilkondaiyur GTFS coordinates are placeholders...
3,555M,3328,Tambaram West Bus Stand,Thiruporur Bus Stand,included,Both endpoints have nearby matching map-ready ...
4,S12,695,Velachery Railway Station,Guindy Railway Station,included_with_warning,Start endpoint approximately 144 m from neares...
5,S54TCT,4580,Porur,Chembarambakkam,included_with_warning,Destination endpoint approximately 989 m from ...


Selected routes: 6
Included without warning: 1
Included with warning: 5


In [54]:
import json
import zipfile
from pathlib import Path
from io import BytesIO

# 1. Locate the original GeoJSON inside the downloaded ZIP
zip_path = Path("../raw/mtc_gtfs.zip")

with zipfile.ZipFile(zip_path, "r") as outer_zip:
    geojson_files = [
        name for name in outer_zip.namelist()
        if name.endswith("all_mtc_routes.geojson")
    ]

    assert len(geojson_files) == 1, (
        f"Expected one all_mtc_routes.geojson file, found {len(geojson_files)}"
    )

    with outer_zip.open(geojson_files[0]) as file:
        original_geojson = json.load(file)

# 2. Create a lookup from the approved route selection
route_lookup = {
    str(row["route_name"]).strip(): row
    for _, row in route_selection.iterrows()
}

# 3. Select and enrich the matching GeoJSON features
selected_features = []

for feature in original_geojson["features"]:
    properties = feature.get("properties", {})
    route_name = str(properties.get("route_name", "")).strip()

    if route_name in route_lookup:
        selection = route_lookup[route_name]

        updated_feature = {
            "type": "Feature",
            "geometry": feature["geometry"],
            "properties": {
                **properties,
                "gtfs_route_id": str(selection["route_id"]),
                "geometry_status": selection["geometry_status"],
                "quality_note": selection["quality_note"]
            }
        }

        selected_features.append(updated_feature)

# 4. Validate the selection
selected_names = {
    str(feature["properties"]["route_name"]).strip()
    for feature in selected_features
}

expected_names = set(route_lookup.keys())

assert selected_names == expected_names, (
    f"Route mismatch. Missing: {expected_names - selected_names}; "
    f"Unexpected: {selected_names - expected_names}"
)

assert len(selected_features) == 6, (
    f"Expected 6 route features, found {len(selected_features)}"
)

# 5. Save the selected GeoJSON
output_dir = Path("../prepared")
output_dir.mkdir(parents=True, exist_ok=True)

selected_geojson = {
    "type": "FeatureCollection",
    "features": selected_features
}

output_path = output_dir / "selected_routes.geojson"

with output_path.open("w", encoding="utf-8") as file:
    json.dump(selected_geojson, file, ensure_ascii=False)

print("Export completed.")
print("Output:", output_path.resolve())
print("Selected route features:", len(selected_features))

for feature in selected_features:
    props = feature["properties"]
    print(
        props["route_name"],
        "| GTFS route ID:", props["gtfs_route_id"],
        "| Status:", props["geometry_status"]
    )

Export completed.
Output: C:\Users\fatim\MTC-Map-Data-Preparation\prepared\selected_routes.geojson
Selected route features: 6
118N | GTFS route ID: 2146 | Status: included_with_warning
120E | GTFS route ID: 2192 | Status: included_with_warning
120KET | GTFS route ID: 2214 | Status: included_with_warning
555M | GTFS route ID: 3328 | Status: included
S12 | GTFS route ID: 695 | Status: included_with_warning
S54TCT | GTFS route ID: 4580 | Status: included_with_warning


In [55]:
stops_path = Path("../prepared/stops_for_map.csv")

assert stops_path.exists(), "stops_for_map.csv was not found."

stops_check = pd.read_csv(stops_path)

assert len(stops_check) == 5419
assert stops_check["stop_id"].is_unique
assert stops_check["map_lat"].notna().all()
assert stops_check["map_lon"].notna().all()

print("Stop dataset verification passed.")
print("Map-ready stops:", len(stops_check))
print("Unique stop IDs:", stops_check["stop_id"].nunique())
print("Missing map coordinates:", stops_check[["map_lat", "map_lon"]].isna().sum().sum())

Stop dataset verification passed.
Map-ready stops: 5419
Unique stop IDs: 5419
Missing map coordinates: 0


In [56]:
from pathlib import Path
import json
import pandas as pd

# 1. Locate prepared files
prepared_dir = Path("../prepared")

stops_path = prepared_dir / "stops_for_map.csv"
routes_path = prepared_dir / "selected_routes.geojson"

# 2. Check file existence
assert stops_path.exists(), f"Missing file: {stops_path.resolve()}"
assert routes_path.exists(), f"Missing file: {routes_path.resolve()}"

print("Both prepared files found.")
print("Stops file:", stops_path.resolve())
print("Routes file:", routes_path.resolve())

# 3. Load stops CSV
stops_df = pd.read_csv(stops_path)

required_stop_columns = {
    "stop_id",
    "stop_name",
    "map_lat",
    "map_lon"
}

missing_columns = required_stop_columns - set(stops_df.columns)

assert not missing_columns, f"Missing stop columns: {missing_columns}"

assert stops_df["stop_id"].is_unique, "Duplicate stop IDs found."
assert stops_df["map_lat"].notna().all(), "Missing latitude values."
assert stops_df["map_lon"].notna().all(), "Missing longitude values."

assert stops_df["map_lat"].between(-90, 90).all()
assert stops_df["map_lon"].between(-180, 180).all()

# 4. Load selected route GeoJSON
with routes_path.open("r", encoding="utf-8") as file:
    routes_geojson = json.load(file)

assert routes_geojson["type"] == "FeatureCollection"

route_features = routes_geojson["features"]

# 5. Validate route geometries and properties
route_summary = []

for feature in route_features:
    properties = feature.get("properties", {})
    geometry = feature.get("geometry")

    assert geometry is not None, "A route has missing geometry."
    assert geometry["type"] in ["LineString", "MultiLineString"]

    assert properties.get("route_name"), "Route name is missing."
    assert properties.get("gtfs_route_id"), "GTFS route ID is missing."
    assert properties.get("geometry_status"), "Geometry status is missing."

    route_summary.append({
        "route_name": properties["route_name"],
        "gtfs_route_id": properties["gtfs_route_id"],
        "geometry_type": geometry["type"],
        "geometry_status": properties["geometry_status"],
        "quality_note": properties.get("quality_note", "")
    })

route_summary_df = pd.DataFrame(route_summary)

# 6. Print verification results
print("\nSTOP DATA")
print("Total map-ready stops:", len(stops_df))
print("Unique stop IDs:", stops_df["stop_id"].nunique())
print("Missing map coordinates:", 
      stops_df[["map_lat", "map_lon"]].isna().sum().sum())

print("\nROUTE DATA")
print("Total route features:", len(route_features))
display(route_summary_df)

# 7. Final checks
assert len(stops_df) == 5419, "Unexpected stop count."
assert len(route_features) == 6, "Expected exactly 6 routes."

expected_routes = {"118N", "120E", "120KET", "555M", "S12", "S54TCT"}

actual_routes = set(route_summary_df["route_name"])

assert actual_routes == expected_routes, (
    f"Route mismatch. Missing: {expected_routes - actual_routes}"
)

print("\nSUCCESS: Prepared map datasets passed structural verification.")

Both prepared files found.
Stops file: C:\Users\fatim\MTC-Map-Data-Preparation\prepared\stops_for_map.csv
Routes file: C:\Users\fatim\MTC-Map-Data-Preparation\prepared\selected_routes.geojson

STOP DATA
Total map-ready stops: 5419
Unique stop IDs: 5419
Missing map coordinates: 0

ROUTE DATA
Total route features: 6


,route_name,gtfs_route_id,geometry_type,geometry_status,quality_note
0,118N,2146,MultiLineString,included_with_warning,Start endpoint approximately 333 m from neares...
1,120E,2192,MultiLineString,included_with_warning,Kilkondaiyur GTFS coordinates are placeholders...
2,120KET,2214,MultiLineString,included_with_warning,Kilkondaiyur GTFS coordinates are placeholders...
3,555M,3328,MultiLineString,included,Both endpoints have nearby matching map-ready ...
4,S12,695,MultiLineString,included_with_warning,Start endpoint approximately 144 m from neares...
5,S54TCT,4580,MultiLineString,included_with_warning,Destination endpoint approximately 989 m from ...



SUCCESS: Prepared map datasets passed structural verification.
